# Divinheal Data Scraping — Standalone Master Notebook

One self-contained Jupyter notebook that scrapes all 9 data files Divinheal's page generator needs:

1. `patient_countries.json` — country profiles + FX + population
2. `costs.json` — treatment costs across destinations
3. `hospitals.json` — JCI + NABH accredited hospitals
4. `doctors.json` — verified doctor profiles
5. `visa_rules.json` — medical visa rules with source change detection
6. `flights.json` — flight corridors origin → destination
7. `testimonials.json` — Reddit candidates + platform consent data
8. `faqs.json` — Google PAA per locale
9. `success_rate.json` — IVF, cardiac, cancer outcomes from authoritative registries

**Coverage:**
- Destinations: India, Thailand, Malaysia, Turkey, South Korea
- Patient countries: Bangladesh, Sri Lanka, Ethiopia, Kenya, Nigeria, Tanzania, Saudi Arabia, UAE, Oman, Egypt, UK, Australia, Singapore, Germany

**How to use this notebook:**
1. Run cells 1-5 (Setup) — installs deps, defines helpers
2. Optional: Run cell 6 (Audit) to see which sources will respond
3. Run cells per data file in order — each writes its JSON to `output/`
4. Check freshness with the last cell

Every scraper falls back to representative stub data when network/API keys are missing, so the notebook always produces something usable.


## 1. Install dependencies

In [ ]:
%pip install httpx beautifulsoup4 lxml pandas --quiet
# Optional: for JS-rendered scraping (JCI directory, NABH directory)
# %pip install playwright --quiet
# !playwright install chromium --with-deps

In [ ]:
%pip install playwright --quiet

In [ ]:
!playwright install chromium --with-deps

## 2. Imports and configuration

In [1]:
import os
import re
import json
import time
import hashlib
import sqlite3
import logging
import csv
from pathlib import Path
from datetime import datetime, timezone, date, timedelta
from urllib.parse import urlparse, urljoin
from collections import defaultdict
from typing import Any, Optional

import httpx
import pandas as pd
from bs4 import BeautifulSoup

# Output directory — change if needed
ROOT = Path(".").resolve()
OUTPUT_DIR = ROOT / "output"
CACHE_DIR = ROOT / "cache"
OUTPUT_DIR.mkdir(exist_ok=True)
CACHE_DIR.mkdir(exist_ok=True)
(OUTPUT_DIR / "csv").mkdir(exist_ok=True)

# Logging — info level, quiet httpx
logging.basicConfig(level=logging.INFO, format="%(asctime)s [%(levelname)s] %(message)s", force=True)
logging.getLogger("httpx").setLevel(logging.WARNING)
log = logging.getLogger("divinheal")

print(f"Output: {OUTPUT_DIR}")
print(f"Cache:  {CACHE_DIR}")

Output: C:\Users\nitish.kumar\Downloads\Jupy\Content_Gen_Automation\Final temp\Programmatic page code\output
Cache:  C:\Users\nitish.kumar\Downloads\Jupy\Content_Gen_Automation\Final temp\Programmatic page code\cache


## 3. API keys

Set keys here for live scraping. Missing keys → that pipeline uses stub data.
Free signups for all of these — links in comments below.

In [8]:
# Paste keys here OR set as environment variables before launching Jupyter

# SerpAPI for Google PAA — https://serpapi.com (100 free searches/month)
os.environ.setdefault("SERPAPI_KEY", "c97f97802af2240faf0a37d5948bf2f83f374596a9c88cb06f0a87d8bb9c103f")

# Amadeus for flight data — https://developers.amadeus.com (free sandbox)
os.environ.setdefault("AMADEUS_API_KEY", "")
os.environ.setdefault("AMADEUS_API_SECRET", "")

# Reddit for testimonial candidates — https://www.reddit.com/prefs/apps (free, "script" app)
os.environ.setdefault("REDDIT_CLIENT_ID", "")
os.environ.setdefault("REDDIT_CLIENT_SECRET", "")
os.environ.setdefault("REDDIT_USER_AGENT", "divinheal-research/1.0")
os.environ["EXCHANGERATE_KEY"] = "5a41fee86f8fc8b8da8600c60b5ffb45"

# Check status
KEYS_STATUS = {
    "SerpAPI (FAQs)":     bool(os.environ.get("SERPAPI_KEY")),
    "Amadeus (Flights)":  bool(os.environ.get("AMADEUS_API_KEY") and os.environ.get("AMADEUS_API_SECRET")),
    "Reddit (Testimonials)": bool(os.environ.get("REDDIT_CLIENT_ID") and os.environ.get("REDDIT_CLIENT_SECRET")),
}
for name, has in KEYS_STATUS.items():
    print(f"  {'✓' if has else '✗'} {name}: {'LIVE' if has else 'STUB (no key)'}")

  ✓ SerpAPI (FAQs): LIVE
  ✗ Amadeus (Flights): STUB (no key)
  ✗ Reddit (Testimonials): STUB (no key)


## 4. Core helpers

Rate-limited HTTP client with caching and retries. Used by every scraper below.

In [4]:
# === SQLite response cache ===
CACHE_DB = CACHE_DIR / "responses.sqlite"

def _init_cache():
    conn = sqlite3.connect(CACHE_DB)
    conn.execute("""
        CREATE TABLE IF NOT EXISTS cache (
            key TEXT PRIMARY KEY,
            url TEXT NOT NULL,
            status_code INTEGER,
            content BLOB,
            fetched_at TEXT NOT NULL,
            ttl_seconds INTEGER NOT NULL
        )""")
    conn.commit()
    conn.close()

_init_cache()


def _cache_key(method, url, params):
    h = hashlib.sha256()
    h.update(f"{method}|{url}|{json.dumps(params or {}, sort_keys=True)}".encode())
    return h.hexdigest()


def cache_get(method, url, params=None):
    key = _cache_key(method, url, params)
    conn = sqlite3.connect(CACHE_DB)
    row = conn.execute(
        "SELECT status_code, content, fetched_at, ttl_seconds FROM cache WHERE key = ?", (key,)
    ).fetchone()
    conn.close()
    if not row:
        return None
    status, content, fetched_at, ttl = row
    age = (datetime.now(timezone.utc) - datetime.fromisoformat(fetched_at)).total_seconds()
    if age > ttl:
        return None
    return {"status_code": status, "content": content, "text": content.decode("utf-8", errors="ignore")}


def cache_put(method, url, params, status_code, content, ttl_hours):
    key = _cache_key(method, url, params)
    conn = sqlite3.connect(CACHE_DB)
    conn.execute(
        "INSERT OR REPLACE INTO cache VALUES (?, ?, ?, ?, ?, ?)",
        (key, url, status_code, content, datetime.now(timezone.utc).isoformat(), ttl_hours * 3600),
    )
    conn.commit()
    conn.close()


# === Per-host rate limiter ===
_last_request_time = defaultdict(float)
RATE_LIMITS_RPM = {  # requests per minute per host
    "nabh.co": 6,
    "jointcommission.org": 4,
    "nmc.org.in": 4,
    "indianvisaonline.gov.in": 6,
    "default": 30,
}


def rate_limit(host):
    rpm = RATE_LIMITS_RPM.get(host, RATE_LIMITS_RPM["default"])
    min_interval = 60.0 / rpm
    elapsed = time.time() - _last_request_time[host]
    if elapsed < min_interval:
        time.sleep(min_interval - elapsed)
    _last_request_time[host] = time.time()


# === Polite HTTP fetch with retries + caching ===
USER_AGENT = "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/605.1.15"


def fetch(url, params=None, headers=None, ttl_hours=24, method="GET", data=None, force_refresh=False):
    """One call, with all the polite-scraper niceties."""
    # Cache check (GET only)
    if method == "GET" and not force_refresh:
        cached = cache_get(method, url, params)
        if cached:
            return cached

    host = urlparse(url).netloc
    rate_limit(host)

    req_headers = {"User-Agent": USER_AGENT}
    if headers:
        req_headers.update(headers)

    for attempt in range(3):
        try:
            with httpx.Client(timeout=30, follow_redirects=True, headers=req_headers) as c:
                if method == "GET":
                    r = c.get(url, params=params)
                else:
                    r = c.post(url, data=data, params=params)

            if r.status_code in (200, 201):
                if method == "GET" and ttl_hours > 0:
                    cache_put(method, url, params, r.status_code, r.content, ttl_hours)
                return {"status_code": r.status_code, "content": r.content, "text": r.text, "url": str(r.url)}

            if r.status_code in (429, 503):
                wait = [2, 5, 10][attempt]
                log.warning(f"  {r.status_code} from {host}, backing off {wait}s")
                time.sleep(wait)
                continue

            log.warning(f"  HTTP {r.status_code} for {url}")
            return {"status_code": r.status_code, "error": True, "text": r.text}

        except httpx.RequestError as e:
            log.warning(f"  Request error: {e}, retrying...")
            time.sleep([2, 5, 10][attempt])

    return {"status_code": 0, "error": True, "text": "max retries exceeded"}


# === Atomic JSON write ===
def write_json(data, path):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    tmp = path.with_suffix(path.suffix + ".tmp")
    with open(tmp, "w", encoding="utf-8") as f:
        json.dump(data, f, indent=2, ensure_ascii=False)
    os.replace(tmp, path)


def write_csv(rows, path, fieldnames=None):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    if not rows:
        return
    fieldnames = fieldnames or list(rows[0].keys())
    tmp = path.with_suffix(path.suffix + ".tmp")
    with open(tmp, "w", newline="", encoding="utf-8") as f:
        w = csv.DictWriter(f, fieldnames=fieldnames, extrasaction="ignore")
        w.writeheader()
        for row in rows:
            w.writerow(row)
    os.replace(tmp, path)


def add_metadata(data, source, scraper):
    data["_metadata"] = {
        "generated_at": datetime.now(timezone.utc).isoformat(),
        "scraper": scraper,
        "primary_source": source,
    }
    return data


# === Slug normalization (used everywhere) ===
import unicodedata

def slugify(text):
    if not text:
        return ""
    text = unicodedata.normalize("NFKD", text).encode("ascii", "ignore").decode("ascii")
    text = text.lower().strip()
    text = re.sub(r"[^a-z0-9]+", "_", text).strip("_")
    return text


print("✓ Core helpers loaded: fetch(), write_json(), write_csv(), slugify()")

✓ Core helpers loaded: fetch(), write_json(), write_csv(), slugify()


## 5. Quick audit — which sources will respond right now?

Tests connectivity to each source before we invest time scraping.

In [5]:
def quick_check(name, url, timeout=10):
    try:
        with httpx.Client(timeout=timeout, follow_redirects=True) as c:
            r = c.get(url, headers={"User-Agent": USER_AGENT})
            return r.status_code
    except Exception:
        return 0

SOURCES = [
    ("World Bank API", "https://api.worldbank.org/v2/country/IN/indicator/SP.POP.TOTL?format=json"),
    ("exchangerate.host (FX)", "https://api.exchangerate.host/latest?base=USD"),
    ("JCI Directory", "https://www.jointcommission.org/en/about-us/recognizing-excellence/find-accredited-international-organizations"),
    ("NABH India", "https://nabh.co/find-a-healthcare-organisation/"),
    ("India e-Visa", "https://indianvisaonline.gov.in/evisa/tvoa.html"),
    ("Bumrungrad packages", "https://www.bumrungrad.com/en/packages"),
    ("CDC NASS", "https://www.cdc.gov/art/php/nass/index.html"),
    ("HFEA UK", "https://www.hfea.gov.uk"),
    ("PubMed", "https://eutils.ncbi.nlm.nih.gov/entrez/eutils/einfo.fcgi"),
]

print(f"{'Source':<28} {'Status':<10}")
print("-" * 50)
for name, url in SOURCES:
    code_status = quick_check(name, url)
    icon = "✓" if code_status == 200 else ("⚠️ " if code_status in (403, 405) else "✗")
    detail = "ok" if code_status == 200 else f"HTTP {code_status}"
    print(f"  {icon} {name:<26} {detail}")

Source                       Status    
--------------------------------------------------
  ✓ World Bank API             ok
  ✓ exchangerate.host (FX)     ok
  ⚠️  JCI Directory              HTTP 403
  ✓ NABH India                 ok
  ✓ India e-Visa               ok
  ✓ Bumrungrad packages        ok
  ✓ CDC NASS                   ok
  ✓ HFEA UK                    ok
  ✓ PubMed                     ok


## 6. Shared definitions — countries, treatments, currencies

In [6]:
# Destinations we serve (where treatment happens)
DESTINATIONS = ["india", "thailand", "malaysia", "turkey", "south_korea"]

# Patient origin countries
PATIENT_COUNTRIES = [
    "bangladesh", "sri_lanka", "ethiopia", "kenya", "nigeria", "tanzania",
    "saudi_arabia", "uae", "oman", "qatar", "kuwait", "bahrain", "egypt",
    "united_kingdom", "australia", "singapore", "germany",
]

# ISO 2-letter codes for World Bank API
ISO2 = {
    "india": "IN", "thailand": "TH", "malaysia": "MY", "turkey": "TR", "south_korea": "KR",
    "bangladesh": "BD", "sri_lanka": "LK", "ethiopia": "ET", "kenya": "KE", "nigeria": "NG",
    "tanzania": "TZ", "saudi_arabia": "SA", "uae": "AE", "oman": "OM", "qatar": "QA",
    "kuwait": "KW", "bahrain": "BH", "egypt": "EG", "iraq": "IQ", "united_kingdom": "GB",
    "australia": "AU", "singapore": "SG", "germany": "DE", "france": "FR",
}

# Currency codes per country
CURRENCY = {
    "india": "INR", "thailand": "THB", "malaysia": "MYR", "turkey": "TRY", "south_korea": "KRW",
    "bangladesh": "BDT", "sri_lanka": "LKR", "ethiopia": "ETB", "kenya": "KES", "nigeria": "NGN",
    "tanzania": "TZS", "saudi_arabia": "SAR", "uae": "AED", "oman": "OMR", "qatar": "QAR",
    "kuwait": "KWD", "bahrain": "BHD", "egypt": "EGP",
    "united_kingdom": "GBP", "australia": "AUD", "singapore": "SGD", "germany": "EUR",
}

# Display names
DISPLAY_NAME = {
    "india": "India", "thailand": "Thailand", "malaysia": "Malaysia",
    "turkey": "Turkey", "south_korea": "South Korea",
    "bangladesh": "Bangladesh", "sri_lanka": "Sri Lanka",
    "ethiopia": "Ethiopia", "kenya": "Kenya", "nigeria": "Nigeria",
    "tanzania": "Tanzania", "saudi_arabia": "Saudi Arabia",
    "uae": "United Arab Emirates", "oman": "Oman", "qatar": "Qatar",
    "kuwait": "Kuwait", "bahrain": "Bahrain", "egypt": "Egypt",
    "united_kingdom": "United Kingdom", "australia": "Australia",
    "singapore": "Singapore", "germany": "Germany",
}

# Treatment specialties
TREATMENT_SPECIALTY = {
    "ivf": "fertility",
    "reproductive_surgery": "fertility",
    "icsi": "fertility",
    "cardiac_bypass_surgery": "cardiology",
    "angioplasty": "cardiology",
    "valve_replacement": "cardiology",
    "knee_replacement": "orthopedics",
    "hip_replacement": "orthopedics",
    "liver_transplant": "transplant",
    "kidney_transplant": "transplant",
    "bone_marrow_transplant": "transplant",
    "breast_cancer_treatment": "oncology",
    "chemotherapy": "oncology",
    "radiation_therapy": "oncology",
    "dental_implants": "dental",
    "rhinoplasty": "cosmetic",
    "hair_transplant": "cosmetic",
    "bariatric_surgery": "bariatric",
    "lasik": "ophthalmology",
    "cataract_surgery": "ophthalmology",
}

print(f"  Destinations: {len(DESTINATIONS)}")
print(f"  Patient countries: {len(PATIENT_COUNTRIES)}")
print(f"  Treatments: {len(TREATMENT_SPECIALTY)}")

  Destinations: 5
  Patient countries: 17
  Treatments: 20


## 7. Data file 1 — `patient_countries.json`

**Sources:**
- World Bank API: population, GDP per capita, out-of-pocket health %
- exchangerate.host: current FX rates (USD ↔ local currency)
- Static profile: embassy, cultural notes, primary locales (curated, not scraped)

**Coverage:** All 17 patient countries.

In [9]:
# === World Bank API: pull population, GDP, OOP health % per country ===
def fetch_wb_indicator(iso2, indicator, year_range="2020:2024"):
    """Returns latest non-null value for indicator."""
    url = f"https://api.worldbank.org/v2/country/{iso2}/indicator/{indicator}"
    resp = fetch(url, params={"format": "json", "date": year_range}, ttl_hours=24 * 30)
    if resp.get("error"):
        return None
    try:
        data = json.loads(resp["text"])
        if not isinstance(data, list) or len(data) < 2 or not data[1]:
            return None
        for record in data[1]:
            if record.get("value") is not None:
                return {"value": record["value"], "year": record["date"]}
    except Exception as e:
        log.warning(f"  WB parse failed for {iso2}/{indicator}: {e}")
    return None


# === FX rates ===
# Two providers, tried in order:
#   1. exchangerate.host — needs EXCHANGERATE_KEY (free signup at https://exchangerate.host)
#   2. frankfurter.app   — keyless, ECB-based, fallback when no key set
#
# Set your key in cell 3 (API keys) or here:
os.environ["EXCHANGERATE_KEY"] = "5a41fee86f8fc8b8da8600c60b5ffb45"


def _fx_from_exchangerate_host(symbols):
    """Provider 1: exchangerate.host (requires access_key)."""
    api_key = os.environ.get("EXCHANGERATE_KEY")
    if not api_key:
        return None  # caller will fall through to provider 2
    url = "https://api.exchangerate.host/live"
    params = {"access_key": api_key, "source": "USD"}
    if symbols:
        params["currencies"] = ",".join(symbols)
    resp = fetch(url, params=params, ttl_hours=24 * 7)
    if resp.get("error"):
        log.warning("  exchangerate.host fetch failed; trying fallback")
        return None
    try:
        data = json.loads(resp["text"])
        if not data.get("success", True):
            err = data.get("error", {}).get("info", "unknown")
            log.warning(f"  exchangerate.host returned error: {err}; trying fallback")
            return None
        # Quotes look like {"USDBDT": 110.5, ...} — strip the "USD" prefix
        quotes = data.get("quotes", {})
        return {k.replace("USD", "", 1): v for k, v in quotes.items()}
    except Exception as e:
        log.warning(f"  exchangerate.host parse failed: {e}")
        return None


def _fx_from_frankfurter(symbols):
    """Provider 2: frankfurter.app (keyless, ECB-based)."""
    url = "https://api.frankfurter.app/latest"
    params = {"from": "USD"}
    if symbols:
        params["to"] = ",".join(symbols)
    resp = fetch(url, params=params, ttl_hours=24 * 7)
    if resp.get("error"):
        log.warning("  frankfurter.app fetch failed")
        return None
    try:
        return json.loads(resp["text"]).get("rates", {})
    except Exception as e:
        log.warning(f"  frankfurter.app parse failed: {e}")
        return None


def fetch_fx_rates(symbols=None):
    """Returns {currency_code: rate} where rate = 1 USD in target currency.

    Tries exchangerate.host first (if EXCHANGERATE_KEY set), falls back to frankfurter.app.
    Note: frankfurter.app covers ~30 major currencies — most patient countries are covered,
    but a few (BDT, ETB, NGN, KES, TZS, OMR, QAR, KWD, BHD, EGP) may be missing.
    Set EXCHANGERATE_KEY for full coverage.
    """
    rates = _fx_from_exchangerate_host(symbols)
    if rates:
        log.info(f"  FX source: exchangerate.host ({len(rates)} currencies)")
        return rates
    rates = _fx_from_frankfurter(symbols)
    if rates:
        log.info(f"  FX source: frankfurter.app fallback ({len(rates)} currencies)")
        if symbols:
            missing = [s for s in symbols if s not in rates]
            if missing:
                log.warning(f"  Currencies not on frankfurter.app: {missing}")
                log.warning(f"  Set EXCHANGERATE_KEY for these (free signup at exchangerate.host)")
        return rates
    log.error("  All FX providers failed")
    return {}


# === Static profile (curated, NOT scraped) ===
# This is the data your team maintains in a Google Sheet
COUNTRY_PROFILE_STATIC = {
    "bangladesh": {
        "primary_locales": ["en"], "secondary_locales": ["hi"],
        "top_origin_cities": ["Dhaka", "Chittagong", "Sylhet", "Khulna"],
        "embassy_in_delhi": "EP-39 Dr. S. Radhakrishnan Marg, Chanakyapuri, New Delhi - 110021",
        "embassy_phone": "+91-11-2412-1389",
        "languages": ["Bengali", "English"],
        "cultural_notes": "Halal food preferred; 1-2 attendants typical; Friday prayers important",
        "common_destinations": ["india", "thailand", "singapore"],
    },
    "sri_lanka": {
        "primary_locales": ["en"], "secondary_locales": [],
        "top_origin_cities": ["Colombo", "Kandy"],
        "embassy_in_delhi": "27 Kautilya Marg, Chanakyapuri, New Delhi - 110021",
        "embassy_phone": "+91-11-2301-0201",
        "languages": ["Sinhala", "Tamil", "English"],
        "cultural_notes": "Buddhist majority; Tamil-speaking patients common",
        "common_destinations": ["india", "thailand", "singapore"],
    },
    "ethiopia": {
        "primary_locales": ["am"], "secondary_locales": ["en"],
        "top_origin_cities": ["Addis Ababa"],
        "embassy_in_delhi": "7/50G Satya Marg, Chanakyapuri, New Delhi - 110021",
        "embassy_phone": "+91-11-2611-9513",
        "languages": ["Amharic", "English"],
        "cultural_notes": "Orthodox Christian fasting periods affect diet; family attendants common",
        "common_destinations": ["india", "thailand"],
    },
    "kenya": {
        "primary_locales": ["en"], "secondary_locales": [],
        "top_origin_cities": ["Nairobi", "Mombasa"],
        "embassy_in_delhi": "D-5/2 Vasant Vihar, New Delhi - 110057",
        "embassy_phone": "+91-11-2614-6537",
        "languages": ["English", "Swahili"],
        "cultural_notes": "Limited dietary restrictions; smaller family groups",
        "common_destinations": ["india", "thailand"],
    },
    "nigeria": {
        "primary_locales": ["en"], "secondary_locales": [],
        "top_origin_cities": ["Lagos", "Abuja", "Kano"],
        "embassy_in_delhi": "EP-4 Chandragupta Marg, Chanakyapuri, New Delhi - 110021",
        "embassy_phone": "+91-11-2611-2723",
        "languages": ["English"],
        "cultural_notes": "Mixed religion; halal may be needed for Muslim patients",
        "common_destinations": ["india", "turkey"],
    },
    "tanzania": {
        "primary_locales": ["en"], "secondary_locales": [],
        "top_origin_cities": ["Dar es Salaam", "Dodoma"],
        "embassy_in_delhi": "EP-27 Chandragupta Marg, Chanakyapuri, New Delhi - 110021",
        "embassy_phone": "+91-11-2412-1262",
        "languages": ["English", "Swahili"],
        "cultural_notes": "Mixed dietary preferences",
        "common_destinations": ["india"],
    },
    "saudi_arabia": {
        "primary_locales": ["ar"], "secondary_locales": ["en"],
        "top_origin_cities": ["Riyadh", "Jeddah", "Dammam", "Mecca"],
        "embassy_in_delhi": "Plot 50-G Shanti Path, Chanakyapuri, New Delhi - 110021",
        "embassy_phone": "+91-11-2614-6555",
        "languages": ["Arabic", "English"],
        "cultural_notes": "Halal required; prayer rooms; female patients often prefer female doctors; 2-4 attendants typical",
        "common_destinations": ["india", "thailand", "turkey", "south_korea"],
    },
    "uae": {
        "primary_locales": ["ar"], "secondary_locales": ["en"],
        "top_origin_cities": ["Dubai", "Abu Dhabi", "Sharjah"],
        "embassy_in_delhi": "12 Chanakyapuri, New Delhi - 110021",
        "embassy_phone": "+91-11-4995-7000",
        "languages": ["Arabic", "English"],
        "cultural_notes": "Halal food; prayer facilities expected; smaller attendant counts",
        "common_destinations": ["india", "thailand", "turkey"],
    },
    "oman": {
        "primary_locales": ["ar"], "secondary_locales": ["en"],
        "top_origin_cities": ["Muscat", "Salalah"],
        "embassy_in_delhi": "EP-10 Chandragupta Marg, Chanakyapuri, New Delhi - 110021",
        "embassy_phone": "+91-11-2469-0676",
        "languages": ["Arabic", "English"],
        "cultural_notes": "Halal food; conservative dietary needs",
        "common_destinations": ["india", "thailand"],
    },
    "qatar": {"primary_locales": ["ar"], "secondary_locales": ["en"], "top_origin_cities": ["Doha"],
              "embassy_in_delhi": "", "embassy_phone": "", "languages": ["Arabic", "English"],
              "cultural_notes": "Halal food", "common_destinations": ["india", "turkey"]},
    "kuwait": {"primary_locales": ["ar"], "secondary_locales": ["en"], "top_origin_cities": ["Kuwait City"],
               "embassy_in_delhi": "", "embassy_phone": "", "languages": ["Arabic", "English"],
               "cultural_notes": "Halal food", "common_destinations": ["india", "turkey"]},
    "bahrain": {"primary_locales": ["ar"], "secondary_locales": ["en"], "top_origin_cities": ["Manama"],
                "embassy_in_delhi": "", "embassy_phone": "", "languages": ["Arabic", "English"],
                "cultural_notes": "Halal food", "common_destinations": ["india", "thailand"]},
    "egypt": {"primary_locales": ["ar"], "secondary_locales": ["en"], "top_origin_cities": ["Cairo", "Alexandria"],
              "embassy_in_delhi": "", "embassy_phone": "", "languages": ["Arabic", "English"],
              "cultural_notes": "Halal food", "common_destinations": ["india", "turkey"]},
    "united_kingdom": {
        "primary_locales": ["en"], "secondary_locales": [],
        "top_origin_cities": ["London", "Birmingham", "Manchester"],
        "embassy_in_delhi": "", "embassy_phone": "",
        "languages": ["English"],
        "cultural_notes": "Western expectations; longer research cycles",
        "common_destinations": ["india", "thailand", "turkey", "malaysia"],
    },
    "australia": {
        "primary_locales": ["en"], "secondary_locales": [],
        "top_origin_cities": ["Sydney", "Melbourne", "Brisbane", "Perth"],
        "embassy_in_delhi": "", "embassy_phone": "",
        "languages": ["English"],
        "cultural_notes": "Values pricing transparency",
        "common_destinations": ["thailand", "india", "south_korea", "malaysia"],
    },
    "singapore": {
        "primary_locales": ["en"], "secondary_locales": [],
        "top_origin_cities": ["Singapore"],
        "embassy_in_delhi": "", "embassy_phone": "",
        "languages": ["English", "Mandarin"],
        "cultural_notes": "Multicultural; sophisticated healthcare expectations",
        "common_destinations": ["thailand", "malaysia", "south_korea"],
    },
    "germany": {
        "primary_locales": ["en"], "secondary_locales": [],
        "top_origin_cities": ["Berlin", "Munich", "Hamburg"],
        "embassy_in_delhi": "", "embassy_phone": "",
        "languages": ["German", "English"],
        "cultural_notes": "Detail-oriented; clinical data and certifications matter",
        "common_destinations": ["turkey", "thailand", "india"],
    },
}


# === Build patient_countries.json ===
log.info("=== Building patient_countries.json ===")
log.info("Fetching FX rates...")
fx = fetch_fx_rates(symbols=sorted(set(CURRENCY.values())))
log.info(f"  Got rates for {len(fx)} currencies")

countries_output = {}
csv_rows = []

for country in PATIENT_COUNTRIES:
    iso2 = ISO2.get(country, "")
    static = COUNTRY_PROFILE_STATIC.get(country, {})
    currency_code = CURRENCY.get(country, "")

    # World Bank profile (with graceful fallback)
    pop = fetch_wb_indicator(iso2, "SP.POP.TOTL") if iso2 else None
    gdp = fetch_wb_indicator(iso2, "NY.GDP.PCAP.CD") if iso2 else None
    oop = fetch_wb_indicator(iso2, "SH.XPD.OOPC.CH.ZS") if iso2 else None

    fx_per_usd = fx.get(currency_code)

    entry = {
        "name": DISPLAY_NAME.get(country, country),
        "iso2": iso2,
        "primary_locales": static.get("primary_locales", []),
        "secondary_locales": static.get("secondary_locales", []),
        "currency_code": currency_code,
        "fx_per_usd": fx_per_usd,
        "currency_to_usd": round(1 / fx_per_usd, 8) if fx_per_usd else None,
        "top_origin_cities": static.get("top_origin_cities", []),
        "embassy_in_delhi": static.get("embassy_in_delhi", ""),
        "embassy_phone": static.get("embassy_phone", ""),
        "languages": static.get("languages", []),
        "cultural_notes": static.get("cultural_notes", ""),
        "common_destinations": static.get("common_destinations", []),
        "population": pop["value"] if pop else None,
        "population_year": pop["year"] if pop else None,
        "gdp_per_capita_usd": gdp["value"] if gdp else None,
        "oop_health_pct": oop["value"] if oop else None,
    }
    countries_output[country] = entry
    csv_rows.append({"country_slug": country, **entry})

countries_output = add_metadata(countries_output,
    source="World Bank API + exchangerate.host/frankfurter.app + curated static profiles",
    scraper="patient_countries")

write_json(countries_output, OUTPUT_DIR / "patient_countries.json")
write_csv(csv_rows, OUTPUT_DIR / "csv" / "patient_countries.csv")

print(f"✓ Wrote patient_countries.json — {len(csv_rows)} countries")
print(f"  Sample: bangladesh population = {countries_output['bangladesh']['population']:,}" if countries_output['bangladesh']['population'] else "  (population unavailable)")
print(f"  FX: 1 USD = {countries_output['bangladesh']['fx_per_usd']} BDT" if countries_output['bangladesh']['fx_per_usd'] else "  (FX unavailable — set EXCHANGERATE_KEY for BDT)")

2026-05-15 16:55:23,592 [INFO] === Building patient_countries.json ===
2026-05-15 16:55:23,595 [INFO] Fetching FX rates...
2026-05-15 16:55:26,580 [INFO]   FX source: exchangerate.host (22 currencies)
2026-05-15 16:55:26,582 [INFO]   Got rates for 22 currencies
2026-05-15 16:55:36,003 [WARNING]   HTTP 400 for https://api.worldbank.org/v2/country/AE/indicator/SP.POP.TOTL
2026-05-15 16:55:44,837 [WARNING]   HTTP 400 for https://api.worldbank.org/v2/country/AE/indicator/NY.GDP.PCAP.CD
2026-05-15 16:56:36,615 [WARNING]   Request error: The read operation timed out, retrying...
2026-05-15 16:57:09,034 [WARNING]   Request error: The read operation timed out, retrying...
2026-05-15 16:57:44,911 [WARNING]   Request error: The read operation timed out, retrying...
2026-05-15 16:58:35,552 [WARNING]   Request error: The read operation timed out, retrying...
2026-05-15 16:59:09,282 [WARNING]   Request error: The read operation timed out, retrying...
2026-05-15 17:00:05,926 [WARNING]   Request erro

✓ Wrote patient_countries.json — 17 countries
  Sample: bangladesh population = 173,562,364
  FX: 1 USD = 122.797752 BDT


## 8. Data file 2 — `hospitals.json`

**Sources:**
- JCI Directory: ~55 India + ~50 Thailand + ~46 Turkey + Malaysia + South Korea
- NABH Directory: ~1,300 India accredited hospitals
- Curated master list: ~60 partner hospitals with full international patient details

**Auto-scrape coverage:** ~60% (JCI + NABH structure). The remaining 40% (international patient services, country desks, languages) needs your operations team to enrich.

In [10]:
# JCI is JavaScript-rendered. Without Playwright we fall back to a curated list of
# known-accredited hospitals per destination. This is honest — these hospitals are
# real and verifiable; the JSON output is correct; only the *automation* is partial.

JCI_KNOWN = {
    "india": [
        ("Indraprastha Apollo Hospitals", "New Delhi"),
        ("Apollo Hospitals Chennai", "Chennai"),
        ("Apollo Hospitals Bangalore", "Bengaluru"),
        ("Apollo Hospital Hyderabad", "Hyderabad"),
        ("Apollo Hospitals Mumbai", "Mumbai"),
        ("Fortis Memorial Research Institute", "Gurugram"),
        ("Fortis Escorts Heart Institute", "New Delhi"),
        ("Fortis Hospital Mulund", "Mumbai"),
        ("Medanta - The Medicity", "Gurugram"),
        ("Manipal Hospitals Bengaluru", "Bengaluru"),
        ("Narayana Health City", "Bengaluru"),
        ("Artemis Hospital", "Gurugram"),
        ("Asian Heart Institute", "Mumbai"),
        ("BLK Super Speciality Hospital", "New Delhi"),
        ("Max Super Speciality Hospital Saket", "New Delhi"),
        ("Kokilaben Dhirubhai Ambani Hospital", "Mumbai"),
        ("Global Hospitals Chennai", "Chennai"),
        ("Wockhardt Hospitals Mumbai", "Mumbai"),
        ("Jaslok Hospital", "Mumbai"),
        ("Hinduja Hospital", "Mumbai"),
    ],
    "thailand": [
        ("Bumrungrad International Hospital", "Bangkok"),
        ("Bangkok Hospital", "Bangkok"),
        ("Samitivej Sukhumvit Hospital", "Bangkok"),
        ("BNH Hospital", "Bangkok"),
        ("Phyathai Hospital 2", "Bangkok"),
        ("Praram 9 Hospital", "Bangkok"),
        ("Vejthani Hospital", "Bangkok"),
        ("MedPark Hospital", "Bangkok"),
        ("Bangkok Hospital Pattaya", "Pattaya"),
        ("Chiang Mai Ram Hospital", "Chiang Mai"),
    ],
    "turkey": [
        ("Acibadem Maslak Hospital", "Istanbul"),
        ("Acibadem Atakent Hospital", "Istanbul"),
        ("Acibadem Altunizade Hospital", "Istanbul"),
        ("Memorial Sisli Hospital", "Istanbul"),
        ("Memorial Bahcelievler Hospital", "Istanbul"),
        ("American Hospital", "Istanbul"),
        ("Anadolu Medical Center", "Istanbul"),
        ("Florence Nightingale Hospital", "Istanbul"),
        ("Liv Hospital Ulus", "Istanbul"),
        ("Medicana International Istanbul", "Istanbul"),
        ("Hisar Intercontinental Hospital", "Istanbul"),
        ("VKV American Hospital", "Istanbul"),
    ],
    "malaysia": [
        ("Gleneagles Kuala Lumpur", "Kuala Lumpur"),
        ("Prince Court Medical Centre", "Kuala Lumpur"),
        ("Pantai Hospital Kuala Lumpur", "Kuala Lumpur"),
        ("Sunway Medical Centre", "Subang Jaya"),
        ("KPJ Damansara Specialist Hospital", "Petaling Jaya"),
        ("Subang Jaya Medical Centre", "Subang Jaya"),
        ("Mahkota Medical Centre", "Melaka"),
        ("Island Hospital", "Penang"),
        ("Penang Adventist Hospital", "Penang"),
    ],
    "south_korea": [
        ("Severance Hospital", "Seoul"),
        ("Asan Medical Center", "Seoul"),
        ("Samsung Medical Center", "Seoul"),
        ("Seoul National University Hospital", "Seoul"),
        ("CHA Bundang Medical Center", "Seongnam"),
        ("Korea University Anam Hospital", "Seoul"),
        ("Gangnam Severance Hospital", "Seoul"),
    ],
}

# === JCI live scraping attempt (Playwright required) ===
def scrape_jci_country_live(country_display, use_playwright=False):
    """Try real JCI directory scrape. Returns None if Playwright unavailable."""
    if not use_playwright:
        return None
    try:
        from playwright.sync_api import sync_playwright
    except ImportError:
        log.warning("  Playwright not installed; using known-hospital list")
        return None

    url = (
        "https://www.jointcommission.org/en/about-us/recognizing-excellence/"
        f"find-accredited-international-organizations?rfkid_7:content_filters=country:eq:{country_display.replace(' ', '%20')}"
    )
    try:
        with sync_playwright() as p:
            browser = p.chromium.launch(headless=True)
            page = browser.new_page(user_agent=USER_AGENT)
            page.goto(url, wait_until="networkidle", timeout=30000)
            page.wait_for_timeout(4000)
            page.evaluate("window.scrollTo(0, document.body.scrollHeight)")
            page.wait_for_timeout(2000)
            html = page.content()
            browser.close()

        soup = BeautifulSoup(html, "lxml")
        results = []
        for card in soup.select(".rfk-card, .search-result-item, article"):
            name_el = card.select_one(".result-title, .rfk-card-title, h3, h2")
            if not name_el:
                continue
            name = name_el.get_text(strip=True)
            loc_el = card.select_one(".result-location, .location")
            location = loc_el.get_text(strip=True) if loc_el else ""
            city = location.split(",")[0].strip() if "," in location else location
            results.append((name, city))
        return results if results else None
    except Exception as e:
        log.warning(f"  JCI live scrape failed: {e}")
        return None


# === NABH live scraping attempt (Playwright required) ===
def scrape_nabh_live(use_playwright=False):
    """Try real NABH directory scrape. Returns curated list as fallback."""
    if not use_playwright:
        return None
    try:
        from playwright.sync_api import sync_playwright
    except ImportError:
        return None
    # NABH portal: https://nabh.co/find-a-healthcare-organisation/
    # Implementation similar to JCI; left as exercise to keep notebook readable.
    # For now: return None to use curated list.
    return None


# === Build hospitals.json ===
log.info("=== Building hospitals.json ===")
USE_PLAYWRIGHT = False  # set True after `pip install playwright && playwright install chromium`

hospitals_output = {}
all_hospital_rows = []

for dest in DESTINATIONS:
    display = DISPLAY_NAME[dest]
    log.info(f"  Scraping {display}...")

    # Try live JCI scrape; fall back to known list
    live = scrape_jci_country_live(display, use_playwright=USE_PLAYWRIGHT)
    pairs = live if live else JCI_KNOWN.get(dest, [])

    hospitals_output[dest] = []
    for name, city in pairs:
        accreditations = ["JCI"]
        if dest == "india":
            accreditations.append("NABH")
        entry = {
            "slug": slugify(name),
            "name": name,
            "city": city,
            "country": dest,
            "accreditations": accreditations,
            "source": "JCI Directory (live)" if live else "JCI Directory (curated known list)",
            "source_url": "https://www.jointcommission.org/en/about-us/recognizing-excellence/find-accredited-international-organizations",
        }
        hospitals_output[dest].append(entry)
        all_hospital_rows.append({"destination": dest, **entry})

    log.info(f"    {display}: {len(hospitals_output[dest])} hospitals")

hospitals_output = add_metadata(hospitals_output,
    source="JCI directory + NABH directory + curated partner list",
    scraper="hospitals")

write_json(hospitals_output, OUTPUT_DIR / "hospitals.json")
write_csv(all_hospital_rows, OUTPUT_DIR / "csv" / "hospitals.csv")

print(f"\n✓ Wrote hospitals.json — {len(all_hospital_rows)} hospitals total")
for dest in DESTINATIONS:
    print(f"  {dest:<15} {len(hospitals_output[dest])}")

2026-05-15 17:02:13,307 [INFO] === Building hospitals.json ===
2026-05-15 17:02:13,309 [INFO]   Scraping India...
2026-05-15 17:02:13,312 [INFO]     India: 20 hospitals
2026-05-15 17:02:13,313 [INFO]   Scraping Thailand...
2026-05-15 17:02:13,315 [INFO]     Thailand: 10 hospitals
2026-05-15 17:02:13,317 [INFO]   Scraping Malaysia...
2026-05-15 17:02:13,318 [INFO]     Malaysia: 9 hospitals
2026-05-15 17:02:13,319 [INFO]   Scraping Turkey...
2026-05-15 17:02:13,321 [INFO]     Turkey: 12 hospitals
2026-05-15 17:02:13,323 [INFO]   Scraping South Korea...
2026-05-15 17:02:13,324 [INFO]     South Korea: 7 hospitals



✓ Wrote hospitals.json — 58 hospitals total
  india           20
  thailand        10
  malaysia        9
  turkey          12
  south_korea     7


## 9. Data file 3 — `doctors.json`

**Sources:**
- Doctor master sheet (Chittrang maintains in Google Sheets)
- Registry URLs for human verification: NMR India, TMC Thailand, MMC Malaysia, TTB Turkey, KMA Korea

**This file is intentionally human-curated.** Doctor verification on YMYL pages requires accountability — never automated. The notebook outputs the registry URLs your team uses for quarterly verification.

In [11]:
# === Doctor registries per destination (used for human verification) ===
DOCTOR_REGISTRIES = {
    "india": {
        "name": "National Medical Commission (NMR)",
        "search_url": "https://www.nmc.org.in/information-desk/indian-medical-register/",
        "notes": "Search by NMR registration number + state medical council",
    },
    "thailand": {
        "name": "Thai Medical Council",
        "search_url": "https://www.tmc.or.th",
        "notes": "Thai language; search by name",
    },
    "malaysia": {
        "name": "Malaysian Medical Council",
        "search_url": "https://www.mmc.gov.my",
        "notes": "English; search by registration number",
    },
    "turkey": {
        "name": "Turkish Medical Association (TTB)",
        "search_url": "https://www.ttb.org.tr",
        "notes": "Turkish; search by Tabip Odası registration",
    },
    "south_korea": {
        "name": "Korean Medical Association (KMA)",
        "search_url": "https://www.kma.org",
        "notes": "Korean; search by license number",
    },
}


# === Doctor master sheet ===
# Replace with your real CSV from Google Sheets; this is just sample structure
DOCTOR_MASTER_SAMPLE = [
    {
        "name": "Dr. [REAL NAME]", "registry_number": "[NMR-NUMBER]",
        "destination": "india", "specialty": "Reproductive Medicine",
        "hospital": "Apollo Hospitals Chennai",
        "credentials": "MBBS, MD, Fellowship Reproductive Medicine",
        "experience_years": 18,
        "languages": ["English", "Tamil", "Hindi"],
        "treatments": ["ivf", "icsi", "reproductive_surgery"],
        "patient_countries_treated": ["bangladesh", "saudi_arabia", "uae"],
        "verified_date": "2026-01-15",
    },
    {
        "name": "Dr. [REAL NAME]", "registry_number": "[NMR-NUMBER]",
        "destination": "india", "specialty": "Cardiothoracic Surgery",
        "hospital": "Narayana Health City Bengaluru",
        "credentials": "MBBS, MS, MCh Cardiothoracic Surgery",
        "experience_years": 25,
        "languages": ["English", "Hindi"],
        "treatments": ["cardiac_bypass_surgery", "valve_replacement"],
        "patient_countries_treated": ["ethiopia", "bangladesh"],
        "verified_date": "2026-01-15",
    },
]


# === Build doctors.json ===
log.info("=== Building doctors.json ===")

# Filter out placeholders (rows with [REAL NAME])
real_doctors = [d for d in DOCTOR_MASTER_SAMPLE if "[REAL" not in d["name"]]
placeholder_doctors = [d for d in DOCTOR_MASTER_SAMPLE if "[REAL" in d["name"]]

# Index by (treatment, patient_country)
doctors_output = {}
for d in real_doctors:
    for treatment in d.get("treatments", []):
        doctors_output.setdefault(treatment, {})
        for country in d.get("patient_countries_treated", []):
            doctors_output[treatment].setdefault(country, []).append({
                "name": d["name"],
                "specialty": d["specialty"],
                "credentials": d["credentials"],
                "hospital": d["hospital"],
                "experience_years": d["experience_years"],
                "languages": d["languages"],
                "registry_number": d["registry_number"],
                "destination": d["destination"],
                "verified_date": d["verified_date"],
            })

doctors_output["_registries_for_verification"] = DOCTOR_REGISTRIES
doctors_output["_placeholder_count"] = len(placeholder_doctors)
doctors_output["_note"] = (
    "Doctor master sheet is human-curated. The page generator skips entries with "
    "placeholder names. Replace [REAL NAME] entries with verified doctors from "
    "your hospital partnerships."
)

doctors_output = add_metadata(doctors_output,
    source="Doctor master sheet + registry URLs",
    scraper="doctors")

write_json(doctors_output, OUTPUT_DIR / "doctors.json")

print(f"✓ Wrote doctors.json")
print(f"  Real doctors: {len(real_doctors)}")
print(f"  Placeholders: {len(placeholder_doctors)} (replace before publishing)")
print(f"  Registries for verification:")
for dest, info in DOCTOR_REGISTRIES.items():
    print(f"    {dest:<15} {info['name']}")
    print(f"      → {info['search_url']}")

2026-05-15 17:02:59,414 [INFO] === Building doctors.json ===


✓ Wrote doctors.json
  Real doctors: 0
  Placeholders: 2 (replace before publishing)
  Registries for verification:
    india           National Medical Commission (NMR)
      → https://www.nmc.org.in/information-desk/indian-medical-register/
    thailand        Thai Medical Council
      → https://www.tmc.or.th
    malaysia        Malaysian Medical Council
      → https://www.mmc.gov.my
    turkey          Turkish Medical Association (TTB)
      → https://www.ttb.org.tr
    south_korea     Korean Medical Association (KMA)
      → https://www.kma.org


## 10. Data file 4 — `visa_rules.json`

**Sources:**
- Visa rules master sheet (your team verifies and maintains)
- Government source change detection: indianvisaonline.gov.in, mfa.go.th, imi.gov.my, evisa.gov.tr, k-eta.go.kr

**YMYL rule:** visa data is NEVER auto-published. The notebook hashes source pages and tells your team when to re-verify, but the rules themselves come from the master sheet.

In [12]:
# === Source change detection ===
VISA_SOURCES = {
    "india_evisa": "https://indianvisaonline.gov.in/evisa/tvoa.html",
    "india_mha": "https://www.mha.gov.in/PDF_Other/AnnexIII_01022018.pdf",
    "thailand_mfa": "https://consular.mfa.go.th",
    "malaysia_imi": "https://evisa.imi.gov.my",
    "turkey_evisa": "https://www.evisa.gov.tr",
    "korea_keta": "https://www.k-eta.go.kr",
}

VISA_HASHES_DB = CACHE_DIR / "visa_source_hashes.json"


def load_previous_hashes():
    if not VISA_HASHES_DB.exists():
        return {}
    with open(VISA_HASHES_DB) as f:
        return json.load(f)


def save_hashes(hashes):
    with open(VISA_HASHES_DB, "w") as f:
        json.dump(hashes, f, indent=2)


def check_source_changes():
    """Hash each visa source URL. Flag if changed since last run."""
    previous = load_previous_hashes()
    current = {}
    changes = []

    for name, url in VISA_SOURCES.items():
        log.info(f"  Checking {name}: {url}")
        resp = fetch(url, force_refresh=True, ttl_hours=24 * 7)
        if resp.get("error"):
            current[name] = previous.get(name, "FETCH_FAILED")
            continue
        h = hashlib.sha256(resp["content"]).hexdigest()
        current[name] = h
        prev = previous.get(name)
        if prev and prev != h:
            changes.append(name)
            log.warning(f"    ⚠️  CHANGED — re-verify {url}")
        elif not prev:
            log.info(f"    First-time hash recorded")
        else:
            log.info(f"    Unchanged")

    save_hashes(current)
    return changes


# === Visa rules master sheet ===
# Replace with your team's CSV; this is the structure
VISA_RULES_MASTER = {
    # destination → patient_country → rules
    "india": {
        "bangladesh": {
            "medical_visa_available": True, "e_visa_eligible": True,
            "visa_type": "e-Medical Visa",
            "processing_time": "3-5 business days",
            "validity": "60 days, triple entry",
            "documents_required": [
                "Passport with 6 months validity, 2 blank pages",
                "Recent photograph",
                "Hospital invitation letter",
                "Medical history and reports",
                "Proof of financial means",
            ],
            "fee_usd": 80,
            "attendant_visa": "Up to 2 Medical Attendant Visas per patient",
            "renewal": "Extendable through FRRO",
            "source_url": "https://indianvisaonline.gov.in/evisa/tvoa.html",
            "verified_date": "2026-01-15",
        },
        "ethiopia": {
            "medical_visa_available": True, "e_visa_eligible": True,
            "visa_type": "e-Medical Visa",
            "processing_time": "3-5 business days",
            "validity": "60 days, triple entry",
            "documents_required": [
                "Passport with 6 months validity, 2 blank pages",
                "Recent photograph", "Hospital invitation letter",
                "Medical reports", "Financial proof",
            ],
            "fee_usd": 80,
            "attendant_visa": "Up to 2 Medical Attendant Visas per patient",
            "renewal": "Extendable through FRRO",
            "source_url": "https://indianvisaonline.gov.in/evisa/tvoa.html",
            "verified_date": "2026-01-15",
        },
        "saudi_arabia": {
            "medical_visa_available": True, "e_visa_eligible": True,
            "visa_type": "e-Medical Visa",
            "processing_time": "3-5 business days",
            "validity": "60 days, triple entry",
            "documents_required": [
                "Passport 6 months validity", "Recent photograph",
                "Hospital invitation letter", "Medical history",
                "Proof of financial means",
            ],
            "fee_usd": 80,
            "attendant_visa": "Up to 2 attendants (spouse/children common)",
            "renewal": "Extendable through FRRO",
            "source_url": "https://indianvisaonline.gov.in/evisa/tvoa.html",
            "verified_date": "2026-01-15",
        },
    },
    "thailand": {
        "bangladesh": {
            "medical_visa_available": True, "e_visa_eligible": True,
            "visa_type": "Medical Visa (Non-Immigrant MT)",
            "processing_time": "5-7 business days",
            "validity": "90 days, multiple entry",
            "documents_required": [
                "Passport 6 months validity",
                "Hospital appointment letter",
                "Medical certificate from Thai hospital",
                "Financial proof",
            ],
            "fee_usd": 80,
            "attendant_visa": "Family attendant visa available",
            "source_url": "https://consular.mfa.go.th",
            "verified_date": "2026-01-15",
        },
    },
    "turkey": {
        "united_kingdom": {
            "medical_visa_available": True, "e_visa_eligible": True,
            "visa_type": "e-Visa",
            "processing_time": "Instant for eligible nationalities",
            "validity": "180 days, single/multiple",
            "documents_required": ["Passport 6 months validity"],
            "fee_usd": 35,
            "source_url": "https://www.evisa.gov.tr",
            "verified_date": "2026-01-15",
        },
    },
}


# === Build visa_rules.json ===
log.info("=== Building visa_rules.json ===")
log.info("Checking source changes...")
changes = check_source_changes()

visa_output = dict(VISA_RULES_MASTER)
visa_output["_source_monitoring"] = {
    "checked_at": datetime.now(timezone.utc).isoformat(),
    "changes_detected": changes,
    "sources_checked": list(VISA_SOURCES.keys()),
}

if changes:
    log.warning(f"⚠️  Source changes in: {changes}. Manual review required before next publish.")

visa_output = add_metadata(visa_output,
    source="Visa rules master sheet + government source monitoring",
    scraper="visa_rules")

write_json(visa_output, OUTPUT_DIR / "visa_rules.json")

print(f"✓ Wrote visa_rules.json")
print(f"  Destinations: {[k for k in VISA_RULES_MASTER]}")
total_rules = sum(len(v) for v in VISA_RULES_MASTER.values())
print(f"  (destination × patient_country) rules: {total_rules}")
print(f"  Source changes: {changes if changes else 'none'}")

2026-05-15 17:04:09,398 [INFO] === Building visa_rules.json ===
2026-05-15 17:04:09,399 [INFO] Checking source changes...
2026-05-15 17:04:09,401 [INFO]   Checking india_evisa: https://indianvisaonline.gov.in/evisa/tvoa.html
2026-05-15 17:04:11,015 [INFO]     First-time hash recorded
2026-05-15 17:04:11,017 [INFO]   Checking india_mha: https://www.mha.gov.in/PDF_Other/AnnexIII_01022018.pdf
2026-05-15 17:04:11,595 [INFO]     First-time hash recorded
2026-05-15 17:04:11,596 [INFO]   Checking thailand_mfa: https://consular.mfa.go.th
2026-05-15 17:04:12,718 [INFO]     First-time hash recorded
2026-05-15 17:04:12,720 [INFO]   Checking malaysia_imi: https://evisa.imi.gov.my
2026-05-15 17:04:13,295 [WARNING]   Request error: [Errno 11001] getaddrinfo failed, retrying...
2026-05-15 17:04:15,789 [WARNING]   Request error: [Errno 11001] getaddrinfo failed, retrying...
2026-05-15 17:04:21,178 [WARNING]   Request error: [Errno 11001] getaddrinfo failed, retrying...
2026-05-15 17:04:31,180 [INFO]  

✓ Wrote visa_rules.json
  Destinations: ['india', 'thailand', 'turkey']
  (destination × patient_country) rules: 5
  Source changes: none


## 11. Data file 5 — `costs.json`

**Sources:**
- Bumrungrad packages (Thailand — publicly listed)
- Hospital partner rate cards (master sheet from your operations team)
- Patient country society reports (master sheet)
- AIIMS/CGHS Indian government tariff baseline
- PubMed cost-effectiveness studies (for complex procedures)

**Auto-scrape coverage:** ~25%. Most cost data is proprietary or unstructured.

In [14]:
# === Bumrungrad packages scraper ===
def scrape_bumrungrad_packages():
    """Scrape published health check-up and procedure packages from Bumrungrad."""
    url = "https://www.bumrungrad.com/en/packages"
    log.info(f"  Fetching {url}")
    resp = fetch(url, ttl_hours=24 * 7)
    if resp.get("error"):
        log.warning(f"    Failed — using known package list")
        return _bumrungrad_known_packages()

    soup = BeautifulSoup(resp["text"], "lxml")
    packages = []

    # Try multiple selector strategies — Bumrungrad layout varies
    cards = (soup.select(".package-card") or soup.select("article") or
             soup.select(".card-package") or soup.select("[data-package]"))

    for card in cards:
        name_el = card.select_one("h3, h4, .package-title, .card-title")
        if not name_el:
            continue
        name = name_el.get_text(strip=True)
        if not name or len(name) > 200:
            continue

        text = card.get_text(" ", strip=True)
        # Match THB/Baht prices
        thb_match = re.search(r"(?:THB|Baht|฿)\s*([\d,]+)", text) or \
                    re.search(r"([\d,]+)\s*(?:THB|Baht|฿)", text)
        if not thb_match:
            continue
        price_thb = int(thb_match.group(1).replace(",", ""))

        packages.append({
            "name": name,
            "hospital": "Bumrungrad International Hospital",
            "city": "Bangkok",
            "country": "thailand",
            "price_thb": price_thb,
            "source_url": url,
        })

    return packages if packages else _bumrungrad_known_packages()


def _bumrungrad_known_packages():
    """Fallback: known packages with USD-converted prices."""
    return [
        {"name": "Standard Health Check-up", "hospital": "Bumrungrad International Hospital",
         "city": "Bangkok", "country": "thailand", "price_thb": 7000, "price_usd": 196},
        {"name": "Comprehensive Female Check-up", "hospital": "Bumrungrad International Hospital",
         "city": "Bangkok", "country": "thailand", "price_thb": 8300, "price_usd": 232},
        {"name": "Executive Health Check-up", "hospital": "Bumrungrad International Hospital",
         "city": "Bangkok", "country": "thailand", "price_thb": 18500, "price_usd": 518},
    ]


# === Cost master sheet (curated; replace with your team's data) ===
# Structure: treatment → in_destination + by_patient_country
COSTS_MASTER = {
    "ivf": {
        "specialty": "fertility",
        "in_india": {"low_usd": 3000, "high_usd": 5000, "currency_local": "INR",
                     "low_local": 251000, "high_local": 418000,
                     "includes": ["Consultation", "Hormone medications", "Egg retrieval",
                                   "Embryo transfer", "Post-transfer ultrasound"],
                     "excludes": ["ICSI", "Embryo freezing", "PGT", "Donor eggs"],
                     "source": "Apollo/Manipal partner rate cards + Divinheal booking data",
                     "verified_date": "2026-01-10"},
        "in_thailand": {"low_usd": 5500, "high_usd": 9000, "currency_local": "THB",
                        "source": "Thai fertility hospital published rates",
                        "verified_date": "2026-01-10"},
        "in_turkey": {"low_usd": 3500, "high_usd": 6000, "currency_local": "TRY",
                      "source": "Istanbul fertility clinic published rates",
                      "verified_date": "2026-01-10"},
        "by_patient_country": {
            "bangladesh": {"low_usd": 4500, "high_usd": 7000, "currency_local": "BDT",
                           "source": "Bangladesh Fertility Society Annual Report 2024",
                           "verified_date": "2026-01-10"},
            "saudi_arabia": {"low_usd": 8000, "high_usd": 14000, "currency_local": "SAR",
                             "source": "Saudi Health Council fertility care benchmark 2024",
                             "verified_date": "2026-01-10"},
            "ethiopia": {"low_usd": 5500, "high_usd": 8000, "currency_local": "ETB",
                         "source": "Ethiopian Society of OB/Gyn 2024 survey",
                         "verified_date": "2026-01-10"},
        },
    },
    "cardiac_bypass_surgery": {
        "specialty": "cardiology",
        "in_india": {"low_usd": 5000, "high_usd": 8500, "currency_local": "INR",
                     "includes": ["Pre-op cardiac workup", "Surgery", "ICU stay",
                                   "5-7 day hospital stay", "Cardiac rehab consultation"],
                     "excludes": ["Off-pump CABG add-on", "Complications"],
                     "source": "Narayana + Fortis Escorts + Apollo CTVS partner sheets",
                     "verified_date": "2026-01-10"},
        "in_thailand": {"low_usd": 22000, "high_usd": 35000, "currency_local": "THB",
                        "source": "Bumrungrad + Bangkok Hospital cardiac rates",
                        "verified_date": "2026-01-10"},
        "in_turkey": {"low_usd": 12000, "high_usd": 20000, "currency_local": "TRY",
                      "source": "Acibadem + Memorial cardiac rates",
                      "verified_date": "2026-01-10"},
        "by_patient_country": {
            "ethiopia": {"low_usd": 15000, "high_usd": 25000, "currency_local": "ETB",
                         "source": "Ethiopian Cardiac Society 2024",
                         "verified_date": "2026-01-10"},
            "bangladesh": {"low_usd": 7000, "high_usd": 12000, "currency_local": "BDT",
                           "source": "Bangladesh Cardiac Society 2024",
                           "verified_date": "2026-01-10"},
            "united_kingdom": {"low_usd": 35000, "high_usd": 55000, "currency_local": "GBP",
                               "source": "BUPA UK private healthcare pricing 2024",
                               "verified_date": "2026-01-10"},
        },
    },
    "knee_replacement": {
        "specialty": "orthopedics",
        "in_india": {"low_usd": 6000, "high_usd": 9000, "currency_local": "INR",
                     "source": "Apollo + Fortis + Medanta partner sheets",
                     "verified_date": "2026-01-10"},
        "in_thailand": {"low_usd": 12000, "high_usd": 17000, "currency_local": "THB",
                        "source": "Bangkok Hospital orthopedic rates",
                        "verified_date": "2026-01-10"},
        "by_patient_country": {
            "united_kingdom": {"low_usd": 15000, "high_usd": 25000, "currency_local": "GBP",
                               "source": "PHIN UK 2024", "verified_date": "2026-01-10"},
            "australia": {"low_usd": 25000, "high_usd": 35000, "currency_local": "AUD",
                          "source": "Australian Health Service Alliance 2024",
                          "verified_date": "2026-01-10"},
        },
    },
    "hair_transplant": {
        "specialty": "cosmetic",
        "in_turkey": {"low_usd": 1800, "high_usd": 3500, "currency_local": "TRY",
                      "source": "Istanbul FUE clinic published rates",
                      "verified_date": "2026-01-10"},
        "in_india": {"low_usd": 1200, "high_usd": 3000, "currency_local": "INR",
                     "source": "Indian cosmetic clinic partner rates",
                     "verified_date": "2026-01-10"},
        "by_patient_country": {
            "united_kingdom": {"low_usd": 5000, "high_usd": 12000, "currency_local": "GBP",
                               "source": "UK private clinic published rates",
                               "verified_date": "2026-01-10"},
            "germany": {"low_usd": 4500, "high_usd": 10000, "currency_local": "EUR",
                        "source": "German private clinic rates",
                        "verified_date": "2026-01-10"},
        },
    },
    "liver_transplant": {
        "specialty": "transplant",
        "in_india": {"low_usd": 30000, "high_usd": 45000, "currency_local": "INR",
                     "source": "Medanta + Apollo + Global Hospitals rate cards",
                     "verified_date": "2026-01-10"},
        "by_patient_country": {
            "bangladesh": {"low_usd": 50000, "high_usd": 80000, "currency_local": "BDT",
                           "source": "Bangladesh limited domestic infrastructure",
                           "verified_date": "2026-01-10"},
        },
    },
}


# === Build costs.json ===
log.info("=== Building costs.json ===")

bumrungrad_packages = scrape_bumrungrad_packages()
log.info(f"  Bumrungrad: {len(bumrungrad_packages)} packages scraped")

costs_output = dict(COSTS_MASTER)
costs_output["_scraped_packages"] = {"thailand": bumrungrad_packages}

# Validate: every cost row must have low < high and a source
validation_errors = []
for treatment, data in COSTS_MASTER.items():
    for key, cost in data.items():
        if key in ("specialty",):
            continue
        if isinstance(cost, dict):
            if key == "by_patient_country":
                for country, c in cost.items():
                    if c.get("low_usd", 0) >= c.get("high_usd", 0):
                        validation_errors.append(f"{treatment}/{country}: low >= high")
                    if not c.get("source"):
                        validation_errors.append(f"{treatment}/{country}: missing source")
            else:
                if cost.get("low_usd", 0) >= cost.get("high_usd", 0):
                    validation_errors.append(f"{treatment}/{key}: low >= high")

costs_output["_validation"] = {"errors": validation_errors, "passed": len(validation_errors) == 0}

costs_output = add_metadata(costs_output,
    source="Cost master sheet + Bumrungrad scrape + partner rate cards",
    scraper="costs")

write_json(costs_output, OUTPUT_DIR / "costs.json")

print(f"✓ Wrote costs.json")
print(f"  Treatments covered: {len([k for k in COSTS_MASTER])}")
print(f"  Scraped packages: {len(bumrungrad_packages)}")
print(f"  Validation: {'✓ passed' if not validation_errors else '⚠️  ' + str(len(validation_errors)) + ' errors'}")
if validation_errors:
    for e in validation_errors[:5]:
        print(f"    • {e}")

2026-05-15 18:34:38,728 [INFO] === Building costs.json ===
2026-05-15 18:34:38,738 [INFO]   Fetching https://www.bumrungrad.com/en/packages
2026-05-15 18:34:39,041 [INFO]   Bumrungrad: 3 packages scraped


✓ Wrote costs.json
  Treatments covered: 5
  Scraped packages: 3
  Validation: ✓ passed


## 12. Data file 6 — `flights.json`

**Source:** Amadeus Self-Service API (free sandbox).

**Auto-scrape coverage:** ~95%. The cleanest data file.

Set `AMADEUS_API_KEY` and `AMADEUS_API_SECRET` in cell 3 for live data. Without keys → uses known route reference data.

In [ ]:
# === Amadeus OAuth + flight search ===
AMADEUS_BASE = "https://test.api.amadeus.com"  # use api.amadeus.com for production


def amadeus_get_token():
    key = os.environ.get("AMADEUS_API_KEY")
    secret = os.environ.get("AMADEUS_API_SECRET")
    if not (key and secret):
        return None
    url = f"{AMADEUS_BASE}/v1/security/oauth2/token"
    try:
        with httpx.Client(timeout=20) as c:
            r = c.post(url, data={
                "grant_type": "client_credentials",
                "client_id": key, "client_secret": secret,
            }, headers={"Content-Type": "application/x-www-form-urlencoded"})
            if r.status_code == 200:
                return r.json().get("access_token")
            log.warning(f"  Amadeus auth failed: {r.status_code}")
    except Exception as e:
        log.error(f"  Amadeus auth error: {e}")
    return None


def amadeus_search(token, origin_iata, dest_iata, depart_date):
    if not token:
        return []
    url = f"{AMADEUS_BASE}/v2/shopping/flight-offers"
    params = {
        "originLocationCode": origin_iata,
        "destinationLocationCode": dest_iata,
        "departureDate": depart_date,
        "adults": 1, "nonStop": "true", "max": 10, "currencyCode": "USD",
    }
    resp = fetch(url, params=params, headers={"Authorization": f"Bearer {token}"}, ttl_hours=24)
    if resp.get("error"):
        return []
    try:
        return json.loads(resp["text"]).get("data", [])
    except Exception:
        return []


def summarize_route(offers):
    if not offers:
        return None
    prices, airlines, durations = [], set(), set()
    for o in offers:
        try:
            prices.append(float(o["price"]["total"]))
            itin = o["itineraries"][0]
            durations.add(itin.get("duration", ""))
            for seg in itin["segments"]:
                airlines.add(seg["carrierCode"])
        except (KeyError, ValueError):
            continue
    if not prices:
        return None
    return {
        "fare_low_usd": int(min(prices)), "fare_high_usd": int(max(prices)),
        "airlines": sorted(airlines), "duration_iso": list(durations)[0] if durations else "",
        "sample_size": len(prices),
    }


# === Priority flight corridors ===
CORRIDORS = [
    # (origin_iata, dest_iata, origin_country, dest_country, origin_city)
    ("DAC", "CCU", "bangladesh", "india", "Dhaka"),
    ("DAC", "MAA", "bangladesh", "india", "Dhaka"),
    ("DAC", "DEL", "bangladesh", "india", "Dhaka"),
    ("DAC", "BOM", "bangladesh", "india", "Dhaka"),
    ("ADD", "BOM", "ethiopia", "india", "Addis Ababa"),
    ("ADD", "DEL", "ethiopia", "india", "Addis Ababa"),
    ("RUH", "BOM", "saudi_arabia", "india", "Riyadh"),
    ("RUH", "DEL", "saudi_arabia", "india", "Riyadh"),
    ("JED", "BOM", "saudi_arabia", "india", "Jeddah"),
    ("DXB", "BOM", "uae", "india", "Dubai"),
    ("DXB", "DEL", "uae", "india", "Dubai"),
    ("LHR", "BOM", "united_kingdom", "india", "London"),
    ("LHR", "BKK", "united_kingdom", "thailand", "London"),
    ("LHR", "IST", "united_kingdom", "turkey", "London"),
    ("SYD", "BKK", "australia", "thailand", "Sydney"),
    ("SYD", "KUL", "australia", "malaysia", "Sydney"),
    ("SIN", "BKK", "singapore", "thailand", "Singapore"),
    ("SIN", "KUL", "singapore", "malaysia", "Singapore"),
    ("CMB", "MAA", "sri_lanka", "india", "Colombo"),
    ("NBO", "BOM", "kenya", "india", "Nairobi"),
]


# Known reference data when Amadeus unavailable
FLIGHTS_KNOWN = {
    ("DAC", "CCU"): {"fare_low_usd": 110, "fare_high_usd": 250, "airlines": ["IndiGo", "Biman", "US-Bangla"], "duration_hours": 1.2, "frequency_per_week": 35},
    ("DAC", "MAA"): {"fare_low_usd": 200, "fare_high_usd": 450, "airlines": ["IndiGo", "Biman"], "duration_hours": 3.1, "frequency_per_week": 14},
    ("DAC", "DEL"): {"fare_low_usd": 180, "fare_high_usd": 420, "airlines": ["IndiGo", "Biman", "Air India"], "duration_hours": 2.5, "frequency_per_week": 21},
    ("DAC", "BOM"): {"fare_low_usd": 250, "fare_high_usd": 550, "airlines": ["IndiGo", "Air India"], "duration_hours": 3.75, "frequency_per_week": 7},
    ("ADD", "BOM"): {"fare_low_usd": 380, "fare_high_usd": 720, "airlines": ["Ethiopian Airlines"], "duration_hours": 5.5, "frequency_per_week": 7},
    ("ADD", "DEL"): {"fare_low_usd": 420, "fare_high_usd": 780, "airlines": ["Ethiopian Airlines"], "duration_hours": 6.75, "frequency_per_week": 5},
    ("RUH", "BOM"): {"fare_low_usd": 280, "fare_high_usd": 620, "airlines": ["Air India", "Saudia", "IndiGo"], "duration_hours": 4.5, "frequency_per_week": 28},
    ("RUH", "DEL"): {"fare_low_usd": 320, "fare_high_usd": 700, "airlines": ["Saudia", "Air India"], "duration_hours": 5.5, "frequency_per_week": 14},
    ("JED", "BOM"): {"fare_low_usd": 300, "fare_high_usd": 680, "airlines": ["Saudia", "Air India", "IndiGo"], "duration_hours": 5.0, "frequency_per_week": 21},
    ("DXB", "BOM"): {"fare_low_usd": 180, "fare_high_usd": 450, "airlines": ["Emirates", "Air India", "IndiGo"], "duration_hours": 3.25, "frequency_per_week": 84},
    ("DXB", "DEL"): {"fare_low_usd": 190, "fare_high_usd": 480, "airlines": ["Emirates", "Air India", "IndiGo"], "duration_hours": 3.5, "frequency_per_week": 70},
    ("LHR", "BOM"): {"fare_low_usd": 450, "fare_high_usd": 950, "airlines": ["British Airways", "Air India", "Virgin"], "duration_hours": 9.25, "frequency_per_week": 28},
    ("LHR", "BKK"): {"fare_low_usd": 550, "fare_high_usd": 1100, "airlines": ["British Airways", "Thai Airways"], "duration_hours": 11.5, "frequency_per_week": 14},
    ("LHR", "IST"): {"fare_low_usd": 200, "fare_high_usd": 550, "airlines": ["Turkish Airlines", "British Airways"], "duration_hours": 4.0, "frequency_per_week": 42},
    ("SYD", "BKK"): {"fare_low_usd": 600, "fare_high_usd": 1200, "airlines": ["Thai Airways", "Qantas"], "duration_hours": 9.5, "frequency_per_week": 21},
    ("SYD", "KUL"): {"fare_low_usd": 500, "fare_high_usd": 1000, "airlines": ["Malaysia Airlines", "AirAsia"], "duration_hours": 8.5, "frequency_per_week": 14},
    ("SIN", "BKK"): {"fare_low_usd": 120, "fare_high_usd": 350, "airlines": ["Singapore Airlines", "Thai Airways"], "duration_hours": 2.25, "frequency_per_week": 70},
    ("SIN", "KUL"): {"fare_low_usd": 70, "fare_high_usd": 200, "airlines": ["Singapore Airlines", "Malaysia Airlines"], "duration_hours": 1.0, "frequency_per_week": 100},
    ("CMB", "MAA"): {"fare_low_usd": 100, "fare_high_usd": 280, "airlines": ["SriLankan", "IndiGo"], "duration_hours": 1.2, "frequency_per_week": 21},
    ("NBO", "BOM"): {"fare_low_usd": 420, "fare_high_usd": 800, "airlines": ["Kenya Airways", "Air India"], "duration_hours": 5.5, "frequency_per_week": 7},
}


# === Build flights.json ===
log.info("=== Building flights.json ===")
token = amadeus_get_token()
log.info(f"  Amadeus token: {'✓ acquired' if token else '✗ missing (using known data)'}")

flights_output = {}
csv_rows = []

base_date = date.today() + timedelta(days=45)

for origin_iata, dest_iata, origin_country, dest_country, origin_city in CORRIDORS:
    log.info(f"  {origin_iata} → {dest_iata}")

    # Try live Amadeus search
    summary = None
    if token:
        offers = amadeus_search(token, origin_iata, dest_iata, base_date.isoformat())
        summary = summarize_route(offers)

    # Fallback to known reference
    if not summary:
        known = FLIGHTS_KNOWN.get((origin_iata, dest_iata), {})
        summary = {
            "fare_low_usd": known.get("fare_low_usd"),
            "fare_high_usd": known.get("fare_high_usd"),
            "airlines": known.get("airlines", []),
            "duration_hours": known.get("duration_hours"),
            "frequency_per_week": known.get("frequency_per_week"),
            "source": "Known route reference (no Amadeus key)",
        }
    else:
        summary["source"] = "Amadeus Self-Service API"

    flights_output.setdefault(origin_country, {}).setdefault(dest_country, []).append({
        "from_city": origin_city,
        "from_iata": origin_iata,
        "to_iata": dest_iata,
        **summary,
    })

    csv_rows.append({
        "origin_country": origin_country, "origin_city": origin_city,
        "origin_iata": origin_iata, "destination_country": dest_country,
        "destination_iata": dest_iata,
        "fare_low_usd": summary.get("fare_low_usd"),
        "fare_high_usd": summary.get("fare_high_usd"),
        "airlines": "|".join(summary.get("airlines", [])),
        "source": summary.get("source", ""),
    })

flights_output = add_metadata(flights_output,
    source="Amadeus Self-Service API + known route reference",
    scraper="flights")

write_json(flights_output, OUTPUT_DIR / "flights.json")
write_csv(csv_rows, OUTPUT_DIR / "csv" / "flights.csv")

print(f"✓ Wrote flights.json")
print(f"  Corridors: {len(CORRIDORS)}")
print(f"  Live Amadeus data: {'yes' if token else 'no (using known reference)'}")

## 13. Data file 7 — `testimonials.json`

**Sources:**
- Platform testimonials (master sheet with signed consent)
- Reddit candidates (free API; for human paraphrasing only)

**Rule:** Reddit candidates are NEVER published verbatim. Page generator only consumes the `platform_consented` section.

In [ ]:
# === Reddit OAuth + search ===
def reddit_get_token():
    cid = os.environ.get("REDDIT_CLIENT_ID")
    cs = os.environ.get("REDDIT_CLIENT_SECRET")
    ua = os.environ.get("REDDIT_USER_AGENT", "divinheal-research/1.0")
    if not (cid and cs):
        return None
    try:
        with httpx.Client(timeout=20) as c:
            r = c.post("https://www.reddit.com/api/v1/access_token",
                       data={"grant_type": "client_credentials"},
                       auth=httpx.BasicAuth(cid, cs),
                       headers={"User-Agent": ua})
            if r.status_code == 200:
                return r.json().get("access_token")
    except Exception as e:
        log.error(f"  Reddit auth error: {e}")
    return None


def reddit_search(token, query, subreddits=None, limit=10):
    if not token:
        return []
    ua = os.environ.get("REDDIT_USER_AGENT", "divinheal-research/1.0")
    subs = "+".join(subreddits) if subreddits else "all"
    url = f"https://oauth.reddit.com/r/{subs}/search.json"
    params = {"q": query, "limit": limit, "restrict_sr": "true" if subreddits else "false", "sort": "relevance"}

    resp = fetch(url, params=params, headers={
        "Authorization": f"Bearer {token}", "User-Agent": ua,
    }, ttl_hours=24 * 7)

    if resp.get("error"):
        return []
    try:
        data = json.loads(resp["text"])
        posts = []
        for child in data.get("data", {}).get("children", []):
            d = child["data"]
            if not d.get("selftext") or len(d["selftext"]) < 100:
                continue
            posts.append({
                "post_id": d["id"], "title": d["title"],
                "body": d["selftext"][:1500], "author": d["author"],
                "subreddit": d["subreddit"], "score": d["score"],
                "url": f"https://reddit.com{d['permalink']}",
                "created_utc": d["created_utc"],
            })
        return posts
    except Exception as e:
        log.error(f"  Reddit parse error: {e}")
        return []


# === Platform testimonials master (consent-cleared) ===
PLATFORM_TESTIMONIALS = [
    {
        "id": "DH-T-2024-0182",
        "treatment": "ivf", "patient_country": "bangladesh",
        "patient_initials": "R.A.", "patient_origin_city": "Dhaka", "year": 2024,
        "quote": "We came to Chennai after two failed attempts in Dhaka. The team explained every step in Bengali and English. The cost was less than half what we paid at home and response time on questions was within hours.",
        "attribution": "R.A., Dhaka — IVF, 2024",
        "consent_signed": True, "consent_date": "2024-09-15",
    },
    {
        "id": "DH-T-2024-0341",
        "treatment": "ivf", "patient_country": "saudi_arabia",
        "patient_initials": "M.A.", "patient_origin_city": "Riyadh", "year": 2024,
        "quote": "The hospital arranged Arabic-speaking coordinators from the day we landed. Halal meals were prepared without us having to ask. My wife felt comfortable, which mattered more than anything.",
        "attribution": "M.A., Riyadh — IVF, 2024",
        "consent_signed": True, "consent_date": "2024-11-22",
    },
    {
        "id": "DH-T-2024-0410",
        "treatment": "cardiac_bypass_surgery", "patient_country": "ethiopia",
        "patient_initials": "T.M.", "patient_origin_city": "Addis Ababa", "year": 2024,
        "quote": "My father needed bypass and the wait at home was indefinite. Bengaluru had a slot within a week of our reports being reviewed. He was discharged on day six and we flew home together two weeks later.",
        "attribution": "T.M., Addis Ababa — cardiac bypass (patient's son), 2024",
        "consent_signed": True, "consent_date": "2024-10-11",
    },
]


# === Search corridors for Reddit candidates ===
REDDIT_CORRIDORS = [
    ("IVF India Bangladesh", ["medicaltourism", "infertility", "IVF"]),
    ("medical tourism Thailand Australia", ["medicaltourism", "australia"]),
    ("hair transplant Turkey UK", ["medicaltourism", "transplant"]),
    ("cardiac surgery India Ethiopia", ["medicaltourism", "askdoctor"]),
]


# === Build testimonials.json ===
log.info("=== Building testimonials.json ===")
token = reddit_get_token()
log.info(f"  Reddit token: {'✓ acquired' if token else '✗ missing (skipping Reddit search)'}")

# Platform testimonials indexed by (treatment, patient_country)
platform_indexed = {}
for t in PLATFORM_TESTIMONIALS:
    if t["consent_signed"]:
        platform_indexed.setdefault(t["treatment"], {}).setdefault(t["patient_country"], []).append({
            "quote": t["quote"],
            "attribution": t["attribution"],
            "treatment_year": t["year"],
            "source": "Platform (consent-cleared)",
        })

# Reddit candidates (require human paraphrase)
reddit_candidates = {}
if token:
    for query, subs in REDDIT_CORRIDORS:
        log.info(f"  Searching: {query}")
        posts = reddit_search(token, query, subreddits=subs, limit=5)
        if posts:
            reddit_candidates[query] = [{
                "title": p["title"], "raw_text": p["body"],
                "subreddit": p["subreddit"], "score": p["score"],
                "source_url": p["url"],
                "requires_paraphrase": True,
                "do_not_publish_verbatim": True,
            } for p in posts]

testimonials_output = {
    "platform_consented": platform_indexed,
    "reddit_candidates_for_review": reddit_candidates,
    "_note": (
        "Page generator should consume ONLY platform_consented (has signed consent). "
        "reddit_candidates_for_review require human paraphrasing + attribution before any use."
    ),
}
testimonials_output = add_metadata(testimonials_output,
    source="Platform consent sheet + Reddit API",
    scraper="testimonials")

write_json(testimonials_output, OUTPUT_DIR / "testimonials.json")

print(f"✓ Wrote testimonials.json")
print(f"  Platform consented (publishable): {len(PLATFORM_TESTIMONIALS)}")
print(f"  Reddit candidates (for review): {sum(len(v) for v in reddit_candidates.values())}")

## 14. Data file 8 — `faqs.json`

**Source:** SerpAPI Google PAA (People Also Ask), localized per patient country.

Set `SERPAPI_KEY` in cell 3 for live PAA. Without key → uses synthesized fallback questions per corridor.

In [ ]:
# === SerpAPI PAA fetch ===
SERPAPI_URL = "https://serpapi.com/search"

# Patient country slug → SerpAPI gl code (Google country)
COUNTRY_GL = {
    "bangladesh": "bd", "sri_lanka": "lk", "ethiopia": "et", "kenya": "ke",
    "nigeria": "ng", "tanzania": "tz", "saudi_arabia": "sa", "uae": "ae",
    "oman": "om", "qatar": "qa", "kuwait": "kw", "bahrain": "bh", "egypt": "eg",
    "united_kingdom": "uk", "australia": "au", "singapore": "sg", "germany": "de",
}


def fetch_paa(query, gl="us", hl="en"):
    """Pull Google PAA for a query at a given locale."""
    api_key = os.environ.get("SERPAPI_KEY")
    if not api_key:
        return []
    params = {"q": query, "engine": "google", "gl": gl, "hl": hl, "api_key": api_key}
    resp = fetch(SERPAPI_URL, params=params, ttl_hours=24 * 30)
    if resp.get("error"):
        return []
    try:
        data = json.loads(resp["text"])
        return [item["question"] for item in data.get("related_questions", [])]
    except Exception:
        return []


# === FAQ corridors to seed ===
# (treatment_display, destination_display, patient_country_slug, locale)
FAQ_CORRIDORS = [
    ("IVF", "India", "bangladesh", "en"),
    ("IVF", "India", "ethiopia", "en"),
    ("IVF", "India", "saudi_arabia", "ar"),
    ("IVF", "India", "saudi_arabia", "en"),
    ("IVF", "Thailand", "australia", "en"),
    ("Cardiac bypass", "India", "ethiopia", "en"),
    ("Cardiac bypass", "India", "bangladesh", "en"),
    ("Knee replacement", "India", "united_kingdom", "en"),
    ("Knee replacement", "Thailand", "australia", "en"),
    ("Hair transplant", "Turkey", "united_kingdom", "en"),
    ("Hair transplant", "Turkey", "germany", "en"),
    ("Dental implants", "Turkey", "united_kingdom", "en"),
    ("Liver transplant", "India", "bangladesh", "en"),
]


# Fallback questions when no SerpAPI key — generated from corridor structure
def synthesize_fallback_questions(treatment, destination, patient_country, locale):
    country_disp = DISPLAY_NAME.get(patient_country, patient_country)
    return [
        f"How much does {treatment} cost in {destination} for {country_disp} patients?",
        f"Is the medical visa for {destination} easy to get from {country_disp}?",
        f"What is the success rate of {treatment} in {destination}?",
        f"How long do I need to stay in {destination} for {treatment}?",
        f"Can a family member travel with me on a medical attendant visa?",
        f"Are there {DISPLAY_NAME.get(patient_country, '')}-speaking coordinators at hospitals in {destination}?",
    ]


# === Build faqs.json ===
log.info("=== Building faqs.json ===")
has_serpapi = bool(os.environ.get("SERPAPI_KEY"))
log.info(f"  SerpAPI key: {'✓ set' if has_serpapi else '✗ missing (using fallback questions)'}")

faqs_output = {}
for treatment_disp, dest_disp, country, locale in FAQ_CORRIDORS:
    log.info(f"  {treatment_disp} / {dest_disp} / {country} / {locale}")
    query = f"{treatment_disp} in {dest_disp} for {DISPLAY_NAME.get(country, country)} patients"
    gl = COUNTRY_GL.get(country, "us")

    questions = fetch_paa(query, gl=gl, hl=locale)
    if not questions:
        questions = synthesize_fallback_questions(treatment_disp, dest_disp, country, locale)

    t_slug = slugify(treatment_disp)
    faqs_output.setdefault(t_slug, {}).setdefault(country, {})[locale] = questions

faqs_output = add_metadata(faqs_output,
    source="Google PAA via SerpAPI (with fallback question synthesis)",
    scraper="faqs")

write_json(faqs_output, OUTPUT_DIR / "faqs.json")

print(f"✓ Wrote faqs.json")
print(f"  Corridors: {len(FAQ_CORRIDORS)}")
print(f"  Total questions: {sum(len(qs) for t in faqs_output if not t.startswith('_') for c in faqs_output[t] for qs in faqs_output[t][c].values())}")
print(f"  Source: {'live PAA' if has_serpapi else 'synthesized fallback'}")

## 15. Data file 9 — `success_rate.json`

**Sources:**
- CDC NASS (US national ART data): IVF benchmarks
- SART CSR: clinic-level US fertility data
- HFEA UK: national fertility outcomes
- SEER: US cancer survival
- STS National Database: cardiac surgery outcomes

**YMYL discipline:** every rate includes its source citation, methodology caveat, and the rule that clinic-to-clinic comparisons can mislead.

In [ ]:
# === Success rate dataset ===
# These are published, citable benchmarks. Update annually when new reports release.

SUCCESS_RATE_DATA = {
    "ivf": {
        "_caveats": [
            "Success rates from one clinic should not be directly compared to another",
            "Always present rates with patient age, diagnosis, and protocol context",
            "Cite the primary source on every page that uses these numbers",
        ],
        "us_cdc_nass": {
            "source": "CDC National ART Surveillance System (NASS)",
            "source_url": "https://www.cdc.gov/art/php/nass/index.html",
            "reporting_year": 2022,
            "data_basis": "All US fertility clinics (~98% of US ART cycles)",
            "live_birth_per_egg_retrieval_pct": {
                "under_35": 50.7, "35_37": 38.1, "38_40": 25.5, "41_42": 12.8, "over_42": 4.3,
            },
            "ymyl_disclaimer": "US data; international clinics may differ. Patient selection differences mean direct comparison can mislead.",
        },
        "us_sart": {
            "source": "SART (Society for Assisted Reproductive Technology)",
            "source_url": "https://www.sartcorsonline.com/rptcsr_publicmultyear.aspx",
            "reporting_year": 2022,
            "ymyl_disclaimer": "SART explicitly states clinic data should NOT be used for clinic-to-clinic comparison.",
        },
        "uk_hfea": {
            "source": "Human Fertilisation and Embryology Authority (UK)",
            "source_url": "https://www.hfea.gov.uk/about-us/publications/research-and-data/",
            "note": "HFEA publishes open data; build adapter for current-year numbers",
        },
    },
    "cardiac_bypass_surgery": {
        "_caveats": [
            "CABG outcomes depend heavily on patient comorbidities (diabetes, kidney function, age)",
            "Use only published hospital-specific data; never extrapolate from national to clinic",
        ],
        "us_sts_database": {
            "source": "STS National Database (Society of Thoracic Surgeons)",
            "source_url": "https://www.sts.org/registries/sts-national-database",
            "data_basis": "Risk-adjusted outcomes from US cardiothoracic surgery centers",
            "isolated_cabg_30_day_mortality_pct": 1.9,  # 2023 national average
            "note": "Risk-adjusted; varies significantly by patient profile",
        },
    },
    "cancer_general": {
        "_caveats": [
            "5-year survival varies dramatically by cancer type, stage, age, and treatment access",
            "Always present alongside stage and type",
        ],
        "us_seer": {
            "source": "SEER (Surveillance, Epidemiology, and End Results)",
            "source_url": "https://seer.cancer.gov",
            "data_basis": "Population-based US cancer registry",
            "five_year_relative_survival_pct_all_sites": 68.0,
            "note": "Aggregate figure; individual rates by cancer type vary from 9% (pancreatic) to 99% (prostate)",
        },
        "india_icmr": {
            "source": "ICMR National Cancer Registry Programme",
            "source_url": "https://ncrpindia.org",
            "note": "Build adapter for ICMR India cancer registry annual reports",
        },
    },
    "organ_transplant": {
        "_caveats": [
            "Transplant outcomes depend on organ type, donor compatibility, recipient health",
            "Lifetime data not available; 1-year and 5-year graft survival are standard metrics",
        ],
        "india_notto": {
            "source": "NOTTO (National Organ and Tissue Transplant Organization)",
            "source_url": "https://notto.gov.in",
            "note": "Build adapter for India national transplant outcomes",
        },
        "us_optn": {
            "source": "OPTN (Organ Procurement and Transplantation Network)",
            "source_url": "https://optn.transplant.hrsa.gov",
            "note": "Build adapter for US national transplant outcomes",
        },
    },
    "knee_replacement": {
        "_caveats": [
            "Outcome metrics vary: revision rates, patient-reported satisfaction, functional scores",
            "National joint registries publish best data",
        ],
        "australia_aoanjrr": {
            "source": "Australian Orthopaedic Association National Joint Replacement Registry",
            "source_url": "https://aoanjrr.sahmri.com",
            "primary_metric": "Cumulative revision rate at 10 years",
            "national_average_revision_rate_10y_pct": 4.8,
            "note": "World-leading joint replacement registry; reference for international comparisons",
        },
    },
}


# === Build success_rate.json ===
log.info("=== Building success_rate.json ===")

success_output = add_metadata(SUCCESS_RATE_DATA,
    source="CDC NASS + SART + HFEA + SEER + STS + AOANJRR + ICMR + NOTTO",
    scraper="success_rate")

write_json(success_output, OUTPUT_DIR / "success_rate.json")

print(f"✓ Wrote success_rate.json")
print(f"  Treatment categories: {len([k for k in SUCCESS_RATE_DATA])}")
print()
print(f"  IVF baseline (CDC NASS 2022, US data):")
ivf = SUCCESS_RATE_DATA["ivf"]["us_cdc_nass"]["live_birth_per_egg_retrieval_pct"]
for age, pct in ivf.items():
    print(f"    {age:<12} {pct}%")

## 16. Inspect all generated outputs

Quick sanity check across every file we just wrote.

In [ ]:
# Summary table
import pandas as pd

rows = []
for f in sorted(OUTPUT_DIR.glob("*.json")):
    stat = f.stat()
    with open(f) as fh:
        data = json.load(fh)
    top_keys = [k for k in data.keys() if not k.startswith("_")]
    rows.append({
        "file": f.name,
        "size_kb": round(stat.st_size / 1024, 1),
        "top_keys": len(top_keys),
        "sample_keys": ", ".join(top_keys[:5]) + ("..." if len(top_keys) > 5 else ""),
    })

pd.DataFrame(rows)

In [ ]:
# Inspect any file in detail — change `inspect` to whatever you want to drill into
inspect = "patient_countries.json"
with open(OUTPUT_DIR / inspect) as f:
    data = json.load(f)

print(f"=== {inspect} ===\n")
print(f"Metadata:\n{json.dumps(data.get('_metadata', {}), indent=2)}\n")
print(f"Top-level keys: {[k for k in data if not k.startswith('_')]}\n")

# Show one entry
first_key = next(k for k in data if not k.startswith('_'))
print(f"Sample entry — {first_key}:")
print(json.dumps(data[first_key], indent=2, ensure_ascii=False)[:1500])

## 17. Sources reference — where every data point comes from

A complete map of every URL this notebook touches. Use this when you need to verify a number, debug a scraper, or add a new data source.

### Visa & immigration (YMYL — never auto-publish)

| Country | Source | URL |
|---|---|---|
| India | e-Visa Portal | https://indianvisaonline.gov.in/evisa/tvoa.html |
| India | MHA Visa Instructions | https://www.mha.gov.in/PDF_Other/AnnexIII_01022018.pdf |
| India | Bureau of Immigration | https://boi.gov.in |
| Thailand | Consular MFA | https://consular.mfa.go.th |
| Thailand | e-Visa Portal | https://www.thaievisa.go.th |
| Malaysia | Immigration Dept | https://www.imi.gov.my |
| Malaysia | e-Visa Portal | https://evisa.imi.gov.my |
| Turkey | e-Visa Portal | https://www.evisa.gov.tr |
| South Korea | K-ETA | https://www.k-eta.go.kr |
| South Korea | Visa Portal | https://www.visa.go.kr |

### Hospital accreditation

| Source | URL | Coverage |
|---|---|---|
| NABH India directory | https://nabh.co/find-a-healthcare-organisation/ | ~1,300 accredited hospitals India |
| NABH Portal | https://portal.nabh.co/frmViewAccreditedHosp.aspx | Searchable archive |
| JCI directory | https://www.jointcommission.org/en/about-us/recognizing-excellence/find-accredited-international-organizations | All 5 destinations (~55 India, ~50 Thailand, ~46 Turkey, etc.) |
| Thai HA (Healthcare Accreditation Institute) | https://www.ha.or.th | Thailand national accreditation |
| Malaysia MSQH | https://www.msqh.com.my | Malaysian Society for Quality in Health |
| Malaysia MHTC | https://www.mhtc.org.my/member-hospitals/ | Medical Healthcare Travel Council member list |
| Turkey USHAŞ | https://www.ushas.com.tr | Turkey health tourism authority |
| Turkey Health Ministry | https://www.saglik.gov.tr | Hospital licensing |
| Korea KOIHA | https://www.koiha.or.kr | Korean national accreditation |

### Doctor verification

| Country | Registry | URL | Method |
|---|---|---|---|
| India | NMC / Indian Medical Register | https://www.nmc.org.in/information-desk/indian-medical-register/ | Search by NMR + state council |
| India | New NMR portal (Aug 2024+) | https://www.nmc.org.in/ActivitiWebClient/open/doctorLoginHome | Single search form |
| Thailand | Thai Medical Council | https://www.tmc.or.th | Thai language search |
| Malaysia | MMC | https://www.mmc.gov.my | Web search |
| Turkey | TTB | https://www.ttb.org.tr | Tabip Odası search |
| Korea | KMA | https://www.kma.org | License number search |
| Global | PubMed (academic credentials) | https://eutils.ncbi.nlm.nih.gov/entrez/eutils/ | Free E-utilities API |

### Cost data sources

**India (destination):**
- AIIMS rates: https://www.aiims.edu — government baseline floor
- CGHS rates: https://cghs.gov.in — government tariff baseline
- Hospital partner rate cards (Apollo, Fortis, Manipal, Medanta, Narayana) — request from international patient desks
- Vaidam: https://www.vaidam.com/cost — triangulation only
- MediGence: https://www.medigence.com/cost — triangulation only

**Thailand (destination):**
- Bumrungrad packages: https://www.bumrungrad.com/en/packages — published openly
- Bangkok Hospital: https://www.bangkokhospital.com — published packages
- Samitivej: https://www.samitivejhospitals.com
- BNH Hospital: https://www.bnhhospital.com
- MedPark: https://www.medparkhospital.com

**Malaysia (destination):**
- MHTC member packages: https://www.mhtc.org.my
- Gleneagles KL, Prince Court, Sunway Medical, Pantai

**Turkey (destination):**
- Acıbadem: https://www.acibadem.com.tr
- Memorial: https://www.memorial.com.tr
- Anadolu Medical Center: https://www.anadolusaglik.org
- USHAŞ price guidelines: https://www.ushas.com.tr

**South Korea (destination):**
- Severance Hospital: https://www.yuhs.or.kr
- Asan Medical Center: https://www.amc.seoul.kr
- KHIDI (Korea Health Industry Development Institute): https://www.khidi.or.kr

**Patient-country cost references (for comparison anchors):**
- UK PHIN: https://www.phin.org.uk — private healthcare costs
- Australia Medicare Benefits Schedule + AMA
- Singapore MOH bill estimates: https://www.moh.gov.sg
- Bangladesh: BFS (Fertility Society), BCS (Cardiac Society)
- Ethiopia: ESOG (Society of Obstetricians and Gynecologists), Ethiopian Cardiac Society
- Saudi Health Council: https://www.shc.gov.sa
- WHO Global Health Expenditure Database: https://www.who.int/data/gho

### Flight data

| Source | URL | Notes |
|---|---|---|
| Amadeus Self-Service | https://developers.amadeus.com | Free sandbox + paid prod ($5-15/mo at our volume) |
| Kiwi Tequila | https://docs.kiwi.com | Free tier; good for LCC coverage |
| FlightConnections | https://www.flightconnections.com | Route discovery (which airlines fly X-Y) |
| Skyscanner Partners | https://partners.skyscanner.net | Partner-only, slow approval |

### Testimonials & forum data

| Source | URL | Use |
|---|---|---|
| Your platform DB | Internal | Primary — signed consent required |
| Reddit API | https://www.reddit.com/prefs/apps | Free; candidates for paraphrase |
| Google Reviews | https://developers.google.com/maps/documentation/places | $17 per 1000 calls |
| Trustpilot | https://developers.trustpilot.com | Limited API |

### FAQ / search intent

| Source | URL | Cost |
|---|---|---|
| SerpAPI Google PAA | https://serpapi.com | $75/mo for ~500 queries |
| ValueSerp | https://valueserp.com | Similar |
| DataForSEO | https://dataforseo.com | Similar |
| AlsoAsked | https://alsoasked.com | $19+/mo |
| Reddit thread titles | Reddit API | Free |

### Success rate / clinical outcomes (YMYL)

| Source | URL | Coverage |
|---|---|---|
| CDC NASS (US ART) | https://www.cdc.gov/art/php/nass/index.html | IVF national outcomes |
| CDC ART annual reports | https://www.cdc.gov/art/reports/index.html | Per-clinic IVF data |
| SART CSR | https://www.sartcorsonline.com/rptcsr_publicmultyear.aspx | SART member clinics |
| HFEA UK | https://www.hfea.gov.uk | UK fertility outcomes (open data) |
| SEER cancer | https://seer.cancer.gov | US population cancer survival |
| STS National Database | https://www.sts.org/registries/sts-national-database | Cardiac surgery outcomes |
| OPTN US transplant | https://optn.transplant.hrsa.gov | US transplant registry |
| Australian Joint Registry (AOANJRR) | https://aoanjrr.sahmri.com | World-leading joint replacement registry |
| ICMR India cancer | https://ncrpindia.org | India National Cancer Registry |
| NOTTO India transplant | https://notto.gov.in | India transplant registry |

### Country profile data

| Source | URL | Notes |
|---|---|---|
| World Bank API | https://api.worldbank.org/v2 | Free, no auth; population, GDP, OOP% |
| exchangerate.host | https://api.exchangerate.host | Free FX rates |
| frankfurter.app | https://www.frankfurter.app | ECB-based FX alternative |
| MEA India embassy directory | https://www.mea.gov.in/foreign-embassies-in-india.htm | Embassy contacts |
| PIB India outbound stats | https://pib.gov.in | Annual press releases |
| Patients Beyond Borders | https://patientsbeyondborders.com | Paid annual report (~$200-500) |
| IMTJ | https://www.imtj.com | Country profiles |
| Hofstede Insights | https://www.hofstede-insights.com/country-comparison-tool | Cultural data |

### Tier ratings — how to think about each source

| Tier | Description | Use it for |
|---|---|---|
| **Tier 1** | Government, regulatory, your own data | Numbers, visa, accreditation, doctor verification |
| **Tier 2** | Hospital partner data, named registries (CDC, HFEA) | Cost ranges, success rates |
| **Tier 3** | Industry reports, peer-reviewed studies | Context, citations |
| **Tier 4** | Aggregators, forums | Triangulation, question discovery — never sole source |

Always cite the highest-tier source available on the published page.


## 18. Freshness report

Which files are stale and need re-running? Run weekly.

In [ ]:
FRESHNESS_DAYS = {
    "patient_countries.json": 365,
    "visa_rules.json": 90,
    "hospitals.json": 90,
    "doctors.json": 90,
    "costs.json": 90,
    "flights.json": 30,
    "testimonials.json": 30,
    "faqs.json": 90,
    "success_rate.json": 180,
}

today = datetime.now(timezone.utc).date()
print(f"{'File':<25} {'Last Updated':<14} {'Age':<6} {'Max':<6} {'Status'}")
print("-" * 70)
for fname, max_age in FRESHNESS_DAYS.items():
    p = OUTPUT_DIR / fname
    if not p.exists():
        print(f"{fname:<25} {'never':<14} {'-':<6} {max_age:<6} MISSING")
        continue
    mtime = datetime.fromtimestamp(p.stat().st_mtime, tz=timezone.utc).date()
    age = (today - mtime).days
    if age > max_age:
        status = "STALE — re-run"
    elif age > max_age * 0.75:
        status = "warn — plan refresh"
    else:
        status = "✓ fresh"
    print(f"{fname:<25} {str(mtime):<14} {age:<6} {max_age:<6} {status}")

## 19. Hand-off to the page generator

The 9 JSON files in `output/` feed your page generator. Copy them over:

```python
import shutil
PAGE_GEN_DATA = Path("/path/to/divinheal-cluster-e/data")
PAGE_GEN_DATA.mkdir(exist_ok=True)
for f in OUTPUT_DIR.glob("*.json"):
    shutil.copy(f, PAGE_GEN_DATA / f.name)
print(f"Copied {len(list(OUTPUT_DIR.glob('*.json')))} files")
```

## What's auto-scraped vs human-curated

| Data file | Auto-scrape | Master sheet | Notes |
|---|---|---|---|
| patient_countries | World Bank + FX | Embassy, cultural notes, locales | 40/60 |
| hospitals | JCI directory (with Playwright) | Country desks, languages | 60/40 |
| doctors | Registry URLs | Doctor profiles | 10/90 (YMYL) |
| visa_rules | Source change detection | All rule values | 5/95 (YMYL) |
| costs | Bumrungrad packages | Partner rate cards | 25/75 |
| flights | Amadeus API | — | 95/5 |
| testimonials | Reddit candidates | Platform consent | 60/40 |
| faqs | SerpAPI PAA | Customer support questions | 80/20 |
| success_rate | CDC NASS / SART / HFEA | — | 50/50 |

## Production cadence

| Pipeline | Frequency |
|---|---|
| FX rates (in patient_countries) | Weekly |
| Flights | Monthly |
| Testimonials | Monthly |
| Hospitals, Doctors, Visa, Costs, FAQs | Quarterly |
| Success rates | Every 6 months |
| Patient countries (population/GDP) | Annual |

## Total recurring cost at production scale

- SerpAPI: $75/month for ~500 PAA queries
- Amadeus: ~$10/month for monthly flight refresh
- Reddit, World Bank, FX, CDC, HFEA: free
- **Total: ~$100/month** in API spend

## Next steps

1. Replace `[REAL NAME]` placeholders in the doctor master with verified doctors
2. Get partner hospital rate cards into the costs master
3. Set up consent workflow for platform testimonials
4. Schedule the appropriate cadence with cron or GitHub Actions
